# CatBoost Depth-4 — Kaggle 최종 제출 (전체 Train 재학습)

**목적:** 기존 3-Stage Ablation Validation에서 가장 높았던 `CAT_depth4` (CatBoost depth=4, iterations=250, LR=0.05, PairLogit)를 **원본 전체 train.csv 기반**으로 제출한다.

- 그룹 A: LightGCN(575 Epoch) + SimpleX(650 Epoch) + UltraGCN(80 Epoch)
- 그룹 B: **SVD 대체 구현** (팀원 실제 ALS·NeuMF·SVD와 다름)
- 그룹 C: **BM25 유사 대체 구현** (팀원 실제 BM25-Shrink와 다름)
- 그룹 T: Masked-item Transformer (Validation 최고 T1 구성, 50 Epoch)
- 그룹 D: **포함되지 않음** (팀원 코드/체크포인트를 받은 적이 없음)

**중요:** 기존 0.046249는 Validation 점수이며 Kaggle Test 점수가 아니다. 이전 Ablation 노트북은 제출 CSV를 만들지 않았다.

**순서:** 대회 원본 데이터 연결 → GPU T4 → Run All → `submission_catboost.csv` 또는 `catboost_submission.zip` 다운로드 → Kaggle Submit Predictions.

아래 코드 셀은 독립 실행 가능하도록 기존 소스 파일들을 복원한다. **Validation ZIP은 필요 없다.** 전체 데이터 기반 내부 holdout으로 재랭커를 학습하고, 모델들을 다시 전체 train으로 학습해 Test 사용자에게 추천한다.

실행은 길 수 있다. 학습 도중 세션 종료에 대비해 결과 저장을 권장한다.

In [ ]:
from pathlib import Path
import importlib.util, sys, torch
WORK=Path('/kaggle/working')
DATA=Path('/kaggle/input')
WORK.mkdir(parents=True,exist_ok=True)
print('Python:',sys.version.split()[0])
print('CUDA available:',torch.cuda.is_available())
print('Input datasets:',[str(p) for p in DATA.iterdir()][:25] if DATA.exists() else '(No Kaggle Input directory)')
for package in ['catboost','sklearn','scipy','pandas','numpy']:
    print(package, bool(importlib.util.find_spec(package)))
assert torch.cuda.is_available(), '최종 전체 재학습용으로 Kaggle GPU T4를 켜고 다시 실행하세요.'
assert importlib.util.find_spec('catboost') is not None, 'CatBoost 모듈을 찾지 못했습니다. Kaggle 노트북에서 !pip install catboost 실행 후 다시 시도하세요.'


## 1. 기존 학습·추론 코드 포함 (별도 `.py` 업로드 불필요)

In [ ]:
%%writefile /kaggle/working/graph_models.py
"""Single-GPU LightGCN / NGCF experiments for the shared Amazon Top-20 split.

Example:
    python graph_models.py --model lightgcn \
      --fit ../amazon_topk_teamkit/validation/val_train.csv \
      --users ../amazon_topk_teamkit/validation/val_users.csv \
      --truth ../amazon_topk_teamkit/validation/val_truth.csv \
      --out lightgcn_val.csv

No metadata or reviews are read. Train interactions are the only graph edges.
"""

from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None


TOP_K = 20
_BaseModel = nn.Module if nn is not None else object


def _id_key(value: str):
    return (0, int(value)) if value.isdecimal() else (1, value)


def _csv_rows(path: str | Path, required: tuple[str, ...]):
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
            raise ValueError(f"{path} must contain {', '.join(required)}")
        for row in reader:
            if None in row:
                raise ValueError(f"Malformed CSV row in {path}")
            yield row


def _identifier(value: str | None, column: str) -> str:
    if not value or value != value.strip() or any(char.isspace() for char in value):
        raise ValueError(f"Invalid {column}: {value!r}")
    return value


@dataclass
class InteractionData:
    user_ids: list[str]
    item_ids: list[str]
    user_index: dict[str, int]
    item_index: dict[str, int]
    positive_users: list[int]
    positive_items: list[int]
    seen_items: dict[int, set[int]]
    target_users: list[str]

    @property
    def num_users(self) -> int:
        return len(self.user_ids)

    @property
    def num_items(self) -> int:
        return len(self.item_ids)

    @property
    def num_edges(self) -> int:
        return len(self.positive_users)


def load_data(fit_path: str | Path, users_path: str | Path) -> InteractionData:
    """Index unique positive edges; do not filter interactions by rating."""
    edges: set[tuple[str, str]] = set()
    for row in _csv_rows(fit_path, ("user_id", "item_id")):
        edges.add((_identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")))
    if not edges:
        raise ValueError("Empty fit interactions")

    target_users: list[str] = []
    target_set: set[str] = set()
    for row in _csv_rows(users_path, ("user_id",)):
        user_id = _identifier(row["user_id"], "user_id")
        if user_id in target_set:
            raise ValueError(f"Duplicate target user: {user_id}")
        target_set.add(user_id)
        target_users.append(user_id)

    train_users = {user for user, _ in edges}
    unseen_users = target_set - train_users
    if unseen_users:
        raise ValueError(f"{len(unseen_users)} target users have no training interaction")
    user_ids = sorted(train_users, key=_id_key)
    item_ids = sorted({item for _, item in edges}, key=_id_key)
    if len(item_ids) <= TOP_K:
        raise ValueError("Need more than 20 known items to recommend 20 unseen items")
    user_index = {value: index for index, value in enumerate(user_ids)}
    item_index = {value: index for index, value in enumerate(item_ids)}
    positive_users: list[int] = []
    positive_items: list[int] = []
    seen_items: dict[int, set[int]] = defaultdict(set)
    for user, item in sorted(edges, key=lambda pair: (_id_key(pair[0]), _id_key(pair[1]))):
        u, i = user_index[user], item_index[item]
        positive_users.append(u)
        positive_items.append(i)
        seen_items[u].add(i)
    for user in target_users:
        if len(seen_items[user_index[user]]) + TOP_K > len(item_ids):
            raise ValueError(f"User {user} has fewer than 20 unseen items")
    return InteractionData(
        user_ids, item_ids, user_index, item_index, positive_users,
        positive_items, dict(seen_items), target_users,
    )


def load_truth(path: str | Path, target_users: Sequence[str]) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    for row in _csv_rows(path, ("user_id", "item_id")):
        user, item = _identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")
        if item in truth[user]:
            raise ValueError(f"Duplicate truth pair: {user}, {item}")
        truth[user].add(item)
    if set(truth) != set(target_users):
        raise ValueError("Truth and target user sets differ")
    return dict(truth)


def recall_at_20(predictions: dict[str, list[str]], truth: dict[str, set[str]]) -> dict:
    """Pooled competition Recall@20: total hits / sum(min(20, |GT_u|))."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user]) for user, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if not denominator:
        raise ValueError("No truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def normalized_graph(data: InteractionData, device):
    """D^-1/2 A D^-1/2 for the undirected user-item graph."""
    if torch is None:
        raise RuntimeError("PyTorch is required to build the graph")
    u = torch.tensor(data.positive_users, dtype=torch.long)
    i = torch.tensor(data.positive_items, dtype=torch.long) + data.num_users
    rows = torch.cat((u, i))
    cols = torch.cat((i, u))
    degree = torch.bincount(rows, minlength=data.num_users + data.num_items).float()
    values = (degree[rows] * degree[cols]).rsqrt()
    indices = torch.stack((rows, cols)).to(device)
    return torch.sparse_coo_tensor(
        indices, values.to(device),
        (data.num_users + data.num_items, data.num_users + data.num_items),
        device=device,
    ).coalesce()


def sparse_edge_dropout(graph, probability: float):
    if probability <= 0:
        return graph
    if not 0 <= probability < 1:
        raise ValueError("Edge dropout must be in [0, 1)")
    keep = torch.rand(graph._nnz(), device=graph.device) >= probability
    return torch.sparse_coo_tensor(
        graph.indices()[:, keep], graph.values()[keep] / (1 - probability),
        graph.shape, device=graph.device,
    ).coalesce()


class LightGCN(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int):
        if nn is None:
            raise RuntimeError("PyTorch is required to train LightGCN")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.layers = layers
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(graph, current)
            layer_outputs.append(current)
        final = torch.stack(layer_outputs, dim=0).mean(dim=0)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


class NGCF(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int, message_dropout: float):
        if nn is None:
            raise RuntimeError("PyTorch is required to train NGCF")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.sum_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.bi_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.message_dropout = message_dropout
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        for layer in (*self.sum_linears, *self.bi_linears):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for sum_linear, bi_linear in zip(self.sum_linears, self.bi_linears):
            neighbors = torch.sparse.mm(graph, current)
            current = F.leaky_relu(sum_linear(current + neighbors) + bi_linear(current * neighbors), negative_slope=0.2)
            current = F.dropout(current, p=self.message_dropout, training=self.training)
            current = F.normalize(current, p=2, dim=1)
            layer_outputs.append(current)
        final = torch.cat(layer_outputs, dim=1)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


def sample_triples(data: InteractionData, count: int, rng: random.Random, device):
    """Sample one uniformly random unseen item per sampled positive edge."""
    if count <= 0:
        raise ValueError("samples-per-epoch must be positive")
    edge_indices = [rng.randrange(data.num_edges) for _ in range(count)]
    users = [data.positive_users[index] for index in edge_indices]
    positive = [data.positive_items[index] for index in edge_indices]
    negative = [rng.randrange(data.num_items) for _ in range(count)]
    for position, (user, item) in enumerate(zip(users, negative)):
        if item in data.seen_items[user]:
            while True:
                candidate = rng.randrange(data.num_items)
                if candidate not in data.seen_items[user]:
                    negative[position] = candidate
                    break
    return (
        torch.tensor(users, dtype=torch.long, device=device),
        torch.tensor(positive, dtype=torch.long, device=device),
        torch.tensor(negative, dtype=torch.long, device=device),
    )


@torch.no_grad() if torch is not None else (lambda function: function)
def predict_top20(model, graph, data: InteractionData, user_batch_size: int) -> dict[str, list[str]]:
    """Score all known items in bounded user batches and mask fit interactions."""
    if user_batch_size <= 0:
        raise ValueError("eval-user-batch-size must be positive")
    model.eval()
    user_vectors, item_vectors = model(graph)
    item_matrix = item_vectors.T.contiguous()
    predictions: dict[str, list[str]] = {}
    for start in range(0, len(data.target_users), user_batch_size):
        batch = data.target_users[start : start + user_batch_size]
        user_indices = torch.tensor([data.user_index[user] for user in batch], dtype=torch.long, device=graph.device)
        scores = user_vectors[user_indices] @ item_matrix
        masked_rows: list[int] = []
        masked_columns: list[int] = []
        for local_row, user_index in enumerate(user_indices.tolist()):
            for item_index in data.seen_items[user_index]:
                masked_rows.append(local_row)
                masked_columns.append(item_index)
        if masked_rows:
            row_index = torch.tensor(masked_rows, dtype=torch.long, device=graph.device)
            column_index = torch.tensor(masked_columns, dtype=torch.long, device=graph.device)
            scores[row_index, column_index] = -float("inf")
        top = torch.topk(scores, k=TOP_K, dim=1).indices.cpu().tolist()
        for user_id, item_indices in zip(batch, top):
            predictions[user_id] = [data.item_ids[index] for index in item_indices]
    return predictions


def write_predictions(path: str | Path, users: Sequence[str], predictions: dict[str, list[str]]) -> None:
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user in users:
            writer.writerow((user, " ".join(predictions[user])))


def _device(name: str):
    if torch is None:
        raise RuntimeError("PyTorch is not installed. Use Kaggle GPU Notebook or install torch.")
    if name == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return torch.device(name)


def _training_loss(model, user_vectors, item_vectors, sampled, regularization: float):
    users, positives, negatives = sampled
    positive_scores = (user_vectors[users] * item_vectors[positives]).sum(dim=1)
    negative_scores = (user_vectors[users] * item_vectors[negatives]).sum(dim=1)
    bpr = F.softplus(negative_scores - positive_scores).mean()
    raw_u = model.user_embedding(users)
    raw_p = model.item_embedding(positives)
    raw_n = model.item_embedding(negatives)
    reg = (raw_u.square().sum(dim=1) + raw_p.square().sum(dim=1) + raw_n.square().sum(dim=1)).mean() / 2
    return bpr + regularization * reg, bpr, reg


def run(args: argparse.Namespace) -> dict:
    device = _device(args.device)
    if args.layers < 1 or args.embedding_dim < 1 or args.epochs < 1:
        raise ValueError("layers, embedding-dim and epochs must be positive")
    if not 0 <= args.edge_dropout < 1 or not 0 <= args.message_dropout < 1:
        raise ValueError("Dropout probabilities must be in [0, 1)")
    if args.reg < 0 or args.lr <= 0:
        raise ValueError("reg must be nonnegative and lr positive")
    torch.manual_seed(args.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
    rng = random.Random(args.seed)
    data = load_data(args.fit, args.users)
    truth = load_truth(args.truth, data.target_users) if args.truth else None
    graph = normalized_graph(data, device)
    if args.model == "lightgcn":
        model = LightGCN(data.num_users, data.num_items, args.embedding_dim, args.layers)
    else:
        model = NGCF(data.num_users, data.num_items, args.embedding_dim, args.layers, args.message_dropout)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    print(json.dumps({
        "model": args.model, "device": str(device), "users": data.num_users,
        "items": data.num_items, "fit_edges": data.num_edges,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
    }), flush=True)

    best_score = -math.inf
    best_state = None
    best_epoch = None
    stale_evaluations = 0
    last_loss = None
    trained_epochs = 0
    for epoch in range(1, args.epochs + 1):
        trained_epochs = epoch
        start = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, args.samples_per_epoch, rng, device)
        epoch_graph = sparse_edge_dropout(graph, args.edge_dropout) if args.model == "ngcf" else graph
        user_vectors, item_vectors = model(epoch_graph)
        loss, bpr, reg = _training_loss(model, user_vectors, item_vectors, sampled, args.reg)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        record = {
            "epoch": epoch, "loss": round(last_loss, 6),
            "bpr": round(float(bpr.detach().cpu()), 6),
            "regularization": round(float(reg.detach().cpu()), 6),
            "seconds": round(time.monotonic() - start, 2),
        }
        del sampled, user_vectors, item_vectors, loss, bpr, reg
        should_evaluate = truth is not None and (epoch == 1 or epoch % args.eval_every == 0 or epoch == args.epochs)
        if should_evaluate:
            predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
            metric = recall_at_20(predictions, truth)
            record.update(metric)
            if metric["recall_at_20"] > best_score:
                best_score = metric["recall_at_20"]
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_evaluations = 0
                record["best"] = True
            else:
                stale_evaluations += 1
                record["best"] = False
        print(json.dumps(record), flush=True)
        if should_evaluate and args.patience > 0 and stale_evaluations >= args.patience:
            print(json.dumps({"early_stop_epoch": epoch}), flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
    write_predictions(args.out, data.target_users, predictions)
    result = {
        "model": args.model, "device": str(device), "fit_edges": data.num_edges,
        "users": len(data.target_users), "output": str(args.out), "last_loss": last_loss,
        "trained_epochs": trained_epochs, "best_epoch": best_epoch,
        "epochs_requested": args.epochs, "seed": args.seed,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
        "learning_rate": args.lr, "regularization": args.reg,
        "edge_dropout": args.edge_dropout if args.model == "ngcf" else 0.0,
        "message_dropout": args.message_dropout if args.model == "ngcf" else 0.0,
    }
    if truth is not None:
        result.update(recall_at_20(predictions, truth))
    print(json.dumps({"final": result}), flush=True)
    return result


def main(argv: Sequence[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="LightGCN / NGCF on the shared Amazon Top-20 split")
    parser.add_argument("--model", choices=("lightgcn", "ngcf"), required=True)
    parser.add_argument("--fit", required=True, help="val_train.csv or full train.csv")
    parser.add_argument("--users", required=True, help="val_users.csv or test.csv")
    parser.add_argument("--truth", help="val_truth.csv; omit for final test predictions")
    parser.add_argument("--out", required=True, help="Prediction CSV")
    parser.add_argument("--metrics-out", help="Optional JSON file for run settings and results")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--samples-per-epoch", type=int, default=100000)
    parser.add_argument("--eval-user-batch-size", type=int, default=256)
    parser.add_argument("--eval-every", type=int, default=5)
    parser.add_argument("--patience", type=int, default=4, help="Validation evaluations without improvement; 0 disables")
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--reg", type=float, default=0.0001)
    parser.add_argument("--edge-dropout", type=float, default=0.0, help="Optional NGCF edge dropout")
    parser.add_argument("--message-dropout", type=float, default=0.1, help="NGCF only")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", help="auto, cpu, cuda:0")
    args = parser.parse_args(argv)
    if args.eval_every < 1 or args.patience < 0:
        parser.error("eval-every must be positive and patience nonnegative")
    result = run(args)
    if args.metrics_out:
        Path(args.metrics_out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/ultra_simplex_runner.py
"""UltraGCN-style and SimpleX-style implicit top-k recommendation.

Designed for the team's provided split and single CUDA GPU (also CPU testable).
Faithfulness notes:
- UltraGCN-style: direct user/item embeddings, degree-aware weighted BCE,
  and item-item co-occurrence constraint. This is a streamlined approximation
  of UltraGCN, not a bit-for-bit reproduction of the published implementation.
- SimpleX-style: user ID + mean item-history tower, cosine contrastive loss,
  leave-one-out history for positive training samples. This is a compact
  implementation of the SimpleX family, not the official codebase.
- Validation positives must NEVER be included in fit history or item neighbors.
"""
from __future__ import annotations

import argparse
import copy
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
from torch import nn
import torch.nn.functional as F

from graph_models import load_data, load_truth, recall_at_20, write_predictions


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_interaction_matrix(data):
    rows = np.asarray(data.positive_users, dtype=np.int64)
    cols = np.asarray(data.positive_items, dtype=np.int64)
    mat = sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                        shape=(data.num_users, data.num_items)).tocsr()
    mat.sum_duplicates()
    mat.data[:] = 1.0
    return mat


def ii_neighbors(mat, k: int):
    """Top-k co-occurrence cosine neighbors from FIT ONLY; padded weights=0."""
    n_items = mat.shape[1]
    ids = np.zeros((n_items, k), dtype=np.int64)
    weights = np.zeros((n_items, k), dtype=np.float32)
    deg = np.maximum(np.asarray(mat.getnnz(axis=0), dtype=np.float32), 1)
    co = (mat.T @ mat).tocsr()
    inv = np.sqrt(deg)
    for i in range(n_items):
        start, end = co.indptr[i], co.indptr[i+1]
        nbrs = co.indices[start:end]
        vals = co.data[start:end]
        if not len(nbrs):
            continue
        scores = vals / (inv[i] * inv[nbrs])
        keep = nbrs != i
        nbrs, scores = nbrs[keep], scores[keep]
        if not len(nbrs):
            continue
        take = min(k, len(nbrs))
        if take < len(nbrs):
            pos = np.argpartition(scores, -take)[-take:]
        else:
            pos = np.arange(len(nbrs))
        pos = pos[np.argsort(-scores[pos], kind='stable')]
        ids[i, :take] = nbrs[pos]
        values = scores[pos].astype(np.float32)
        weights[i, :take] = values / max(values.sum(), 1e-9)
    return ids, weights


class UltraStyle(nn.Module):
    def __init__(self, n_users, n_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user_emb.weight, std=.05)
        nn.init.normal_(self.item_emb.weight, std=.05)

    def vectors(self, mat=None):
        return self.user_emb.weight, self.item_emb.weight


class SimpleXStyle(nn.Module):
    def __init__(self, n_users, n_items, dim, gamma):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.gamma = gamma
        nn.init.normal_(self.user_emb.weight, std=.05)
        nn.init.normal_(self.item_emb.weight, std=.05)

    def vectors(self, mat):
        # CSR FIT-history aggregation only; do not include held-out labels.
        history = torch.sparse_csr_tensor(
            torch.from_numpy(mat.indptr.astype(np.int64)).to(self.item_emb.weight.device),
            torch.from_numpy(mat.indices.astype(np.int64)).to(self.item_emb.weight.device),
            torch.from_numpy(mat.data.astype(np.float32)).to(self.item_emb.weight.device),
            mat.shape, device=self.item_emb.weight.device)
        lengths = torch.tensor(np.maximum(mat.getnnz(axis=1), 1),
                               dtype=torch.float32, device=self.item_emb.weight.device)
        agg = torch.sparse.mm(history, self.item_emb.weight) / lengths[:, None]
        return (F.normalize(self.user_emb.weight + self.gamma * agg, dim=-1),
                F.normalize(self.item_emb.weight, dim=-1))


def history_sum_for_batch(item_emb, flat_hist, rowptr, users):
    """Differentiable ragged history sum, for selected users only."""
    starts = rowptr[users]
    lengths = rowptr[users + 1] - starts
    offsets = torch.cat([torch.zeros((1,), device=users.device, dtype=torch.long),
                         lengths.cumsum(0)[:-1]])
    owner = torch.repeat_interleave(torch.arange(len(users), device=users.device), lengths)
    flat_positions = torch.repeat_interleave(starts, lengths) + (
        torch.arange(owner.numel(), device=users.device) - offsets[owner]
    )
    bag_idx = flat_hist[flat_positions]
    sums = F.embedding_bag(bag_idx, item_emb.weight, offsets,
                           mode='sum', include_last_offset=False)
    return sums, lengths


def sample_batch(users_all, items_all, n_items, observed_codes, batch, n_negs):
    dev = users_all.device
    sampled = torch.randint(len(users_all), (batch,), device=dev)
    users = users_all[sampled]
    positives = items_all[sampled]
    negatives = torch.randint(n_items, (batch, n_negs), device=dev)
    for _ in range(15):
        ids = users[:, None] * n_items + negatives
        pos = torch.searchsorted(observed_codes, ids.reshape(-1)).clamp_max(len(observed_codes)-1)
        mask = (observed_codes[pos].reshape_as(ids) == ids)
        if not mask.any().item():
            break
        negatives[mask] = torch.randint(n_items, (int(mask.sum().item()),), device=dev)
    else:
        raise RuntimeError('Unable to sample unobserved negatives')
    return users, positives, negatives


def evaluate(model, data, mat, truth, device, batch_size=128):
    model.eval()
    predictions = {}
    with torch.no_grad():
        user_vectors, item_vectors = model.vectors(mat)
        item_trans = item_vectors.T.contiguous()
        for start in range(0, len(data.target_users), batch_size):
            batch_users = data.target_users[start:start+batch_size]
            uidx = [data.user_index[u] for u in batch_users]
            scores = user_vectors[torch.tensor(uidx, device=device)] @ item_trans
            for r, ui in enumerate(uidx):
                seen = data.seen_items.get(ui, set())
                if seen:
                    scores[r, torch.tensor(list(seen), dtype=torch.long,device=device)] = -float('inf')
            top = scores.topk(20, dim=1, sorted=True).indices.cpu().tolist()
            for user, idxs in zip(batch_users, top):
                predictions[user] = [data.item_ids[i] for i in idxs]
        score = recall_at_20(predictions, truth) if truth else None
    return predictions, score


def run(a):
    assert a.epochs > 0 and a.eval_every > 0 and a.batch_size > 0 and a.steps_per_epoch > 0
    assert a.negatives > 0 and a.lr > 0 and a.reg >= 0
    device = torch.device(a.device)
    seed_all(a.seed)
    data = load_data(a.fit, a.users)
    truth = load_truth(a.truth, data.target_users) if a.truth else None
    mat = get_interaction_matrix(data)
    if a.model == 'ultragcn':
        model = UltraStyle(data.num_users, data.num_items, a.dim).to(device)
        user_degree = torch.tensor(np.maximum(mat.getnnz(axis=1), 1), dtype=torch.float32, device=device)
        item_degree = torch.tensor(np.maximum(mat.getnnz(axis=0), 1), dtype=torch.float32, device=device)
        if a.ii_weight > 0:
            ii_id, ii_wt = ii_neighbors(mat, a.ii_topk)
            ii_id = torch.from_numpy(ii_id).to(device)
            ii_wt = torch.from_numpy(ii_wt).to(device)
        else:
            ii_id, ii_wt = None, None
    elif a.model == 'simplex':
        model = SimpleXStyle(data.num_users, data.num_items, a.dim, a.gamma).to(device)
        rowptr = torch.tensor(mat.indptr.astype(np.int64), device=device)
        flat_history = torch.tensor(mat.indices.astype(np.int64), device=device)
    else:
        raise ValueError('Unexpected model')
    optim = torch.optim.Adam(model.parameters(), lr=a.lr)
    u_edges = torch.tensor(data.positive_users, dtype=torch.long, device=device)
    i_edges = torch.tensor(data.positive_items, dtype=torch.long, device=device)
    observed = (u_edges * data.num_items + i_edges).sort().values.unique(sorted=True)

    best_epoch = None
    best_score = -1.0
    best_state = None
    stale = 0
    last_loss = None
    for epoch in range(1, a.epochs+1):
        begin = time.time()
        model.train()
        loss_acc = 0.0
        for _ in range(a.steps_per_epoch):
            u, i, j = sample_batch(u_edges, i_edges, data.num_items, observed,
                                   a.batch_size, a.negatives)
            optim.zero_grad(set_to_none=True)
            if a.model == 'ultragcn':
                uv = model.user_emb(u)
                pv = model.item_emb(i)
                nv = model.item_emb(j)
                pos = (uv * pv).sum(-1)
                neg = (uv[:,None,:] * nv).sum(-1)
                # Degree-aware confidence for user-item constraint.
                confidence = (user_degree[u] / item_degree[i]).sqrt().clamp(0.25, 4.0)
                bce_pos = (1.0 + a.beta * confidence) * F.softplus(-pos)
                bce_neg = F.softplus(neg).mean(-1) * a.neg_weight
                loss = (bce_pos + bce_neg).mean()
                if ii_id is not None:
                    nbr_emb = model.item_emb(ii_id[i])
                    nbr_s = (uv[:,None,:] * nbr_emb).sum(-1)
                    item_loss = (F.softplus(-nbr_s) * ii_wt[i]).sum(-1).mean()
                    loss = loss + a.ii_weight * item_loss
                reg = .5*(uv.square().sum(-1).mean()+pv.square().sum(-1).mean()+nv.square().sum(-1).mean())
            else:
                # Leave the current target item OUT of the user's training history.
                history_sum, lengths = history_sum_for_batch(model.item_emb, flat_history, rowptr, u)
                agg_loo = (history_sum - model.item_emb(i)) / (lengths-1).clamp_min(1)[:,None]
                user_r = F.normalize(model.user_emb(u) + model.gamma * agg_loo, dim=-1)
                p = F.normalize(model.item_emb(i), dim=-1)
                n = F.normalize(model.item_emb(j), dim=-1)
                pos_s = (user_r * p).sum(-1)
                neg_s = (user_r[:,None,:] * n).sum(-1)
                loss = ((1.0 - pos_s) + a.neg_weight * F.relu(neg_s-a.margin).square().mean(-1)).mean()
                reg = .5*(model.user_emb(u).square().sum(-1).mean()+model.item_emb(i).square().sum(-1).mean()+model.item_emb(j).square().sum(-1).mean())
            loss = loss + a.reg * reg
            if not torch.isfinite(loss).item():
                raise RuntimeError('Nonfinite loss')
            loss.backward()
            optim.step()
            loss_acc += loss.item()
        last_loss = loss_acc / a.steps_per_epoch
        if (epoch % a.eval_every == 0 or epoch == a.epochs or epoch == 1) and truth:
            preds, result = evaluate(model, data, mat, truth, device, a.eval_batch)
            best = result['recall_at_20'] > best_score
            if best:
                best_score = result['recall_at_20']
                best_epoch = epoch
                best_state = copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()})
                stale = 0
            else:
                stale += 1
            print(json.dumps({'epoch':epoch,'loss':last_loss,'recall_at_20':result['recall_at_20'],
                              'hits':result['hits'],'best':best,'seconds':round(time.time()-begin,2)}),flush=True)
            if a.patience>0 and stale>=a.patience:
                break
        elif epoch % max(a.eval_every, 10) == 0 or epoch == a.epochs:
            print(json.dumps({'epoch':epoch,'loss':last_loss,'seconds':round(time.time()-begin,2)}),flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions, result = evaluate(model, data, mat, truth, device, a.eval_batch)
    write_predictions(a.out, data.target_users, predictions)
    output = dict(model=a.model, out=a.out, fit=a.fit, best_epoch=best_epoch,
                  trained_epochs=epoch, epochs_requested=a.epochs, last_loss=last_loss,
                  dim=a.dim, lr=a.lr, negatives=a.negatives, reg=a.reg,
                  steps_per_epoch=a.steps_per_epoch, batch_size=a.batch_size,
                  ii_weight=a.ii_weight, beta=a.beta, gamma=a.gamma,
                  margin=a.margin, neg_weight=a.neg_weight, seed=a.seed)
    if result:
        output.update(result)
    if a.metrics:
        Path(a.metrics).write_text(json.dumps(output,indent=2),encoding='utf-8')
    print(json.dumps({'final':output}), flush=True)
    return output


def main():
    p=argparse.ArgumentParser(description='UltraGCN-style and SimpleX-style collaborative filtering')
    p.add_argument('--model', choices=['ultragcn','simplex'], required=True)
    p.add_argument('--fit', required=True)
    p.add_argument('--users', required=True)
    p.add_argument('--truth', default=None)
    p.add_argument('--out', required=True)
    p.add_argument('--metrics', default=None)
    p.add_argument('--epochs',type=int,default=70)
    p.add_argument('--eval-every',type=int,default=10)
    p.add_argument('--patience',type=int,default=5)
    p.add_argument('--device',default='cuda:0')
    p.add_argument('--dim',type=int,default=128)
    p.add_argument('--batch-size',type=int,default=2048)
    p.add_argument('--steps-per-epoch',type=int,default=40)
    p.add_argument('--negatives',type=int,default=10)
    p.add_argument('--lr',type=float,default=.002)
    p.add_argument('--reg',type=float,default=1e-5)
    p.add_argument('--ii-weight',type=float,default=.1)
    p.add_argument('--ii-topk',type=int,default=10)
    p.add_argument('--beta',type=float,default=.1)
    p.add_argument('--neg-weight',type=float,default=4.)
    p.add_argument('--gamma',type=float,default=1.)
    p.add_argument('--margin',type=float,default=.2)
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--eval-batch',type=int,default=128)
    a=p.parse_args()
    run(a)


if __name__=='__main__':
    main()


In [ ]:
%%writefile /kaggle/working/team_allsteps_model_pipeline.py
#!/usr/bin/env python3
"""Amazon top-k | model-driven three-stage experimental pipeline.

Stage 1: Train/predict with native model implementations or team model adapters;
         group ensembles, weighted RRF and complementary-hit diagnostics.
Stage 2: Train a small masked-item Transformer on available interactions.
Stage 3: Train LGBMRanker + CatBoostRanker on INNER holdout features,
         evaluate on untouched SHARED validation truth.

NO team-member prediction CSV input is required. The A model runners generate
internal temporary CSVs because they were originally file-based implementations.
B and C are reproducible *proxy baselines*, NOT faithful teammates' code.
D is an optional fit_predict adapter, skipped unless provided by the user.
No Kaggle test submission is produced here.
"""
from __future__ import annotations
import argparse, collections, csv, hashlib, importlib.util, json, math, os, random, shutil, subprocess, sys, time, traceback, zipfile
from pathlib import Path
from typing import Dict, List

import numpy as np
from scipy import sparse
from scipy.sparse.linalg import svds
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAMES = ['A','B','C','D','T']


def atomic_json(path, d):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    p = path.with_suffix(path.suffix + '.tmp')
    p.write_text(json.dumps(d, ensure_ascii=False, indent=2, default=str), encoding='utf8')
    p.replace(path)


def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()


def locate_teamkit(shared_zip, root, work):
    """Locate the exact shared validation split in a ZIP OR an extracted Kaggle Dataset.

    Kaggle Add Input can expose uploaded ZIPs as unpacked directories:
        /kaggle/input/datasets/<owner>/<dataset>/amazon_topk_teamkit/validation/
    Earlier code incorrectly searched ONLY *.zip, causing FileNotFoundError.
    """
    root=Path(root)
    dest=Path(work)/'teamkit_unpack'
    dest.mkdir(parents=True,exist_ok=True)
    wanted_id='f0dfad2f4431a876'
    required=['val_train.csv','val_truth.csv','val_users.csv','manifest.json']

    def directory_candidates(base):
        # Accept a validation directory itself, its parent, or any Kaggle subdirectory.
        base=Path(base)
        raw=[base,base/'validation',base/'amazon_topk_teamkit'/'validation']
        if base.is_dir():
            raw.extend(p.parent for p in base.rglob('manifest.json'))
        seen=set()
        for cand in raw:
            s=str(cand)
            if s in seen:continue
            seen.add(s)
            if not cand.is_dir() or not all((cand/x).is_file() for x in required):continue
            try:
                meta=json.loads((cand/'manifest.json').read_text(encoding='utf8'))
            except Exception:continue
            if meta.get('split_id')==wanted_id:
                for toolkit in (cand.parent/'teamkit.py',cand/'teamkit.py',cand.parent.parent/'teamkit.py'):
                    if toolkit.is_file():
                        yield cand,toolkit
                        break

    selected_dir=None
    selected_zip=None
    if shared_zip:
        source=Path(shared_zip)
        if not source.exists():
            raise FileNotFoundError(f'SHARED_ZIP 경로가 존재하지 않습니다: {source}')
        if source.is_dir():
            selected_dir=next(directory_candidates(source),None)
            if selected_dir is None:
                raise FileNotFoundError(f'SHARED_ZIP 폴더에 분할 ID {wanted_id}인 validation/teamkit.py가 없습니다: {source}')
        elif source.is_file() and zipfile.is_zipfile(source):
            selected_zip=source
        else:
            raise ValueError(f'SHARED_ZIP은 올바른 팀 공통 ZIP 또는 폴더여야 합니다: {source}')
    else:
        # Unpacked Kaggle dataset comes first (user's earlier notebook had this layout).
        selected_dir=next(directory_candidates(root),None)
        if selected_dir is None:
            for candidate in sorted(root.rglob('*.zip')):
                if candidate.is_file() and candidate.suffix.lower()=='.zip':
                    try:
                        with zipfile.ZipFile(candidate) as z:
                            names=z.namelist()
                            if any(n.endswith('validation/manifest.json') for n in names) and any(n.endswith('teamkit.py') for n in names):
                                selected_zip=candidate
                                break
                    except (OSError,zipfile.BadZipFile):continue

    if selected_dir is None and selected_zip is None:
        raise FileNotFoundError(
            '팀 공통 Validation이 보이지 않습니다. Kaggle Add Input에서 '
            'swaidataset/amazon-topk-teamkit을 연결하세요. '
            'ZIP 또는 압축 해제된 validation 폴더를 모두 지원합니다.'
        )

    if selected_dir is not None:
        val_dir, source_module=selected_dir
        (dest/'validation').mkdir(parents=True,exist_ok=True)
        for filename in required:
            shutil.copyfile(val_dir/filename,dest/'validation'/filename)
        shutil.copyfile(source_module,dest/'teamkit.py')
        print('✅ Team validation: extracted folder',val_dir,flush=True)
    else:
        source=selected_zip
        with zipfile.ZipFile(source) as z:
            names=z.namelist()
            for suffix in ['teamkit.py',*[f'validation/{x}' for x in required]]:
                matches=[n for n in names if n==suffix or n.endswith('/'+suffix)]
                if not matches:
                    raise FileNotFoundError(f'공통 팀 ZIP 내부에 {suffix} 파일이 없습니다: {source}')
                target=dest/suffix
                target.parent.mkdir(parents=True,exist_ok=True)
                # Only write known member names to known target paths, avoid Zip Slip.
                with z.open(matches[0]) as fi,target.open('wb') as fo:
                    shutil.copyfileobj(fi,fo)
        print('✅ Team validation: ZIP',source,flush=True)

    sys.path.insert(0,str(dest))
    # Ensure repeated notebook runs use freshly copied source, not a stale module.
    sys.modules.pop('teamkit',None)
    import teamkit
    manifest=teamkit.verify_split(dest/'validation')
    if manifest.get('split_id')!=wanted_id:
        raise RuntimeError(f'Validation split mismatch: expected={wanted_id}, got={manifest.get("split_id")}')
    print('✅ Verified split',wanted_id,'fit=',manifest.get('fit_rows'),'truth=',manifest.get('truth_rows'),flush=True)
    return teamkit,dest/'validation',manifest

def load_interactions(path):
    seen=collections.defaultdict(set)
    events=collections.defaultdict(list)
    pop=collections.Counter()
    with open(path,newline='',encoding='utf8') as fh:
        for r in csv.DictReader(fh):
            u=str(r['user_id']);i=str(r['item_id'])
            if i in seen[u]:continue
            seen[u].add(i);pop[i]+=1
            try:t=float(r.get('timestamp') or 0)
            except ValueError:t=0
            events[u].append((i,t))
    for u in events: events[u].sort(key=lambda x:(x[1],x[0]))
    return dict(seen),dict(events),pop


def inner_split(original, fitpath, truthpath, userspath, seed=47, holdout=.18):
    """Create an extra TRAIN-only holdout. NEVER use shared val_truth as ranker labels."""
    rg=np.random.default_rng(seed)
    seen,events,pop=load_interactions(original)
    # Avoid carrying official validation labels into reranker training.
    with open(fitpath,'w',newline='',encoding='utf8') as a, open(truthpath,'w',newline='',encoding='utf8') as b, open(userspath,'w',newline='',encoding='utf8') as c:
        wa=csv.writer(a); wb=csv.writer(b);wc=csv.writer(c)
        wa.writerow(['user_id','item_id','timestamp']);wb.writerow(['user_id','item_id']);wc.writerow(['user_id'])
        for u,items in sorted(events.items()):
            if len(items)<2:
                for i,t in items:wa.writerow([u,i,int(t)])
                continue
            n=min(10,len(items)-1,max(1,int(round(len(items)*holdout))))
            removed=set(map(int,rg.choice(len(items),size=n,replace=False)))
            wc.writerow([u])
            for k,(i,t) in enumerate(items):
                if k in removed: wb.writerow([u,i])
                else: wa.writerow([u,i,int(t)])
    return fitpath,truthpath,userspath


def top_unseen(uid, ranked, seen, fallback, k=20):
    excluded=seen.get(uid,set()); chosen=[];used=set()
    for i in list(ranked)+list(fallback):
        i=str(i)
        if i not in excluded and i not in used:
            chosen.append(i);used.add(i)
            if len(chosen)>=k:break
    if len(chosen)<k:raise RuntimeError(f'Insufficient unseen items ({len(chosen)} < {k}) for {uid}')
    return chosen


def score_pred(pred,truth,k=20):
    hits=sum(len(set(pred.get(u,[])[:k]) & items) for u,items in truth.items())
    denom=sum(min(k,len(x)) for x in truth.values())
    return dict(recall_at_20=hits/denom if denom else 0,hits=hits,denominator=denom,users=len(truth))


def validate_pred(pred,users,seen,catalog,k=20):
    expected=set(users)
    if set(pred)!=expected:
        raise ValueError(f'User mismatch predictions={len(pred)} expected={len(expected)} missing={list(expected-set(pred))[:3]}')
    for u,row in pred.items():
        if len(row)!=k or len(set(row))!=k or (set(row)&seen.get(u,set())) or not set(row).issubset(catalog):
            raise ValueError(f'Bad predictions for user={u}, row={row[:5]}')


def write_users(path,users):
    with open(path,'w',newline='',encoding='utf8') as f:
        w=csv.writer(f);w.writerow(['user_id']);w.writerows([[u] for u in users])


def write_pred(path,pred,users):
    with open(path,'w',newline='',encoding='utf8') as f:
        w=csv.writer(f);w.writerow(['user_id','item_id']);w.writerows([[u,' '.join(pred[u])] for u in users])


def read_pred(path):
    with open(path,newline='',encoding='utf8') as f:
        return {str(r['user_id']):r['item_id'].split() for r in csv.DictReader(f)}


def native_A_runners(fit,users_path,out,profile,device,seed,script_dir):
    """The user's ACTUAL A-group scripts; outputs are ephemeral compatibility artifacts."""
    epochs={
        'SMOKE':(2,2,2,128,3),
        'PILOT':(100,130,50,20000,10),
        'FULL':(575,650,80,100000,35),
    }[profile]
    lg_epochs,simplex_epochs,ultra_epochs,samples,steps=epochs
    specs={
     'lightgcn': (script_dir/'graph_models.py', [
         '--model','lightgcn','--embedding-dim','128','--layers','3','--lr','0.003','--reg','0.0001',
         '--samples-per-epoch',str(samples),'--eval-every','25','--patience','0','--epochs',str(lg_epochs)]),
     'simplex': (script_dir/'ultra_simplex_runner.py',[
         '--model','simplex','--dim','128','--lr','0.002','--reg','0.00001','--negatives','20',
         '--gamma','0.5','--margin','0.2','--neg-weight','4','--batch-size','2048',
         '--steps-per-epoch',str(steps),'--eval-every','25','--patience','0','--epochs',str(simplex_epochs)]),
     'ultragcn': (script_dir/'ultra_simplex_runner.py',[
         '--model','ultragcn','--dim','128','--lr','0.001','--reg','0.00001','--negatives','10',
         '--ii-weight','0.3','--beta','0.1','--neg-weight','4','--batch-size','2048',
         '--steps-per-epoch',str(steps),'--eval-every','25','--patience','0','--epochs',str(ultra_epochs)])
    }
    out.mkdir(exist_ok=True,parents=True)
    res={}
    for name,(script,args) in specs.items():
        pred_path=out/(name+'_native_predictions.csv')
        met=out/(name+'.json'); log=out/(name+'.log');stamp=out/(name+'.stamp.json')
        key={'fit_hash':sha256(fit),'spec':args,'script_hash':sha256(script),'seed':seed,'device':device}
        if pred_path.exists() and stamp.exists():
            try:
                if json.loads(stamp.read_text())==key:
                    res[name]=read_pred(pred_path)
                    print('Cache restored:',name,flush=True); continue
            except Exception:pass
        extra=['--fit',str(fit),'--users',str(users_path),'--out',str(pred_path),
               '--seed',str(seed),'--device',device]
        if name=='lightgcn':extra.extend(['--metrics-out',str(met)])
        else:extra.extend(['--metrics',str(met)])
        cmd=[sys.executable,str(script),*args,*extra]
        print('Training native',name,'...',flush=True)
        with log.open('w',encoding='utf8') as fo:
            p=subprocess.run(cmd,stdout=fo,stderr=subprocess.STDOUT, text=True)
        if p.returncode:
            print(log.read_text(encoding='utf8')[-3000:])
            raise RuntimeError(f'{name} failed; inspect {log}')
        res[name]=read_pred(pred_path)
        atomic_json(stamp,key)
    return res


def rrf_fuse(members:dict,weights:dict,users:list,seen:dict,fallback:list,k=60,outk=20):
    out={}
    for u in users:
        scores=collections.defaultdict(float)
        for name,w in weights.items():
            if w<=0 or name not in members: continue
            for rank,i in enumerate(members[name].get(u,[]),1):
                scores[i]+=float(w)/(k+rank)
        ranked=sorted(scores,key=lambda i:(-scores[i],i))
        out[u]=top_unseen(u,ranked,seen,fallback,outk)
    return out


def all_scores(users,seen,pop,fallback,predictions,truth,out):
    rows=[]
    for name,pred in predictions.items():
        validate_pred(pred,users,seen,set(pop),20)
        rows.append({'model':name,**score_pred(pred,truth)})
    write_table(out/'single_scores.csv',rows)
    return rows


def write_table(path,rows):
    if not rows:return
    path.parent.mkdir(exist_ok=True,parents=True)
    with open(path,'w',newline='',encoding='utf8') as f:
        w=csv.DictWriter(f,fieldnames=list(dict.fromkeys(k for row in rows for k in row)))
        w.writeheader();w.writerows(rows)


def group_ABLATION(members,users,seen,pop,truth,out,extra_T=False):
    fallback=[i for i,_ in pop.most_common()]
    combos={
       'A+C_50_50':{'A':.50,'C':.50},
       'A+B+C':{'A':.40,'B':.20,'C':.40},
       'D+A_60_40':{'D':.60,'A':.40},
       'D+C_60_40':{'D':.60,'C':.40},
       'D+B_70_30':{'D':.70,'B':.30},
       'D+A+C_50_25_25':{'D':.50,'A':.25,'C':.25},
       'D+A+C_60_20_20':{'D':.60,'A':.20,'C':.20},
       'D+A+C+B':{'D':.45,'A':.25,'C':.20,'B':.10},
       'A+B+C+D_equal':{'A':.25,'B':.25,'C':.25,'D':.25},
       'A+T_50_50':{'A':.5,'T':.5},
       'C+T_50_50':{'C':.5,'T':.5},
       'A+C+T':{'A':.35,'C':.25,'T':.40},
       'A+B+C+T':{'A':.3,'B':.1,'C':.3,'T':.3},
       'D+A+C+T':{'D':.4,'A':.2,'C':.2,'T':.2},
    }
    rows=[]; best_pred=None;best_score=-1
    for name,weights in combos.items():
        if not set(weights).issubset(members):continue
        if 'T' in weights and not extra_T:continue
        for k in [20,60]:
            pred=rrf_fuse(members,weights,users,seen,fallback,k=k)
            sc=score_pred(pred,truth)
            row={'model':f'{name}_k{k}','type':'RRF','weights':json.dumps(weights), 'k':k,**sc}
            rows.append(row)
            if sc['recall_at_20']>best_score:best_score=sc['recall_at_20'];best_pred=pred
    rows=sorted(rows,key=lambda x:-x['recall_at_20'])
    write_table(out/('ensemble_with_transformer.csv' if extra_T else 'ensemble_step1.csv'),rows)
    atomic_json(out/('best_with_transformer.json' if extra_T else 'best_step1.json'),rows[0] if rows else {})
    return rows,best_pred


def complementarity(members,truth,out):
    rows=[]
    hitset={name:{(u,i) for u,ts in truth.items() for i in set(pred.get(u,[]))&ts} for name,pred in members.items()}
    un={}
    for name,s in hitset.items():
        others=set().union(*(s2 for n,s2 in hitset.items() if n!=name)) if len(hitset)>1 else set()
        un[name]={'model':name,'hits':len(s),'unique_hits':len(s-others)}
    write_table(out/'unique_hits.csv',list(un.values()))
    names=list(members)
    for a_idx,a in enumerate(names):
        for b in names[a_idx+1:]:
            s1=hitset[a];s2=hitset[b]
            rows.append({'a':a,'b':b,'shared_hits':len(s1&s2),'union_hits':len(s1|s2),
                         'unique_to_a':len(s1-s2),'unique_to_b':len(s2-s1)})
    write_table(out/'pairwise_hits.csv',rows)
    oracle=set().union(*hitset.values()) if hitset else set()
    atomic_json(out/'oracle.json',{'oracle_hits':len(oracle),
                                  'oracle_recall_at_20':len(oracle)/sum(min(20,len(ts)) for ts in truth.values())})


def make_sparse(seen,catalog,users):
    uids=list(users);items=sorted(catalog,key=lambda x:(len(x),x))
    uid={u:i for i,u in enumerate(uids)};iid={i:j for j,i in enumerate(items)}
    rr=[];cc=[]
    for u,row in seen.items():
        if u not in uid:continue
        for i in row:
            if i in iid:rr.append(uid[u]);cc.append(iid[i])
    X=sparse.csr_matrix((np.ones(len(rr),dtype=np.float32),(rr,cc)),shape=(len(uids),len(items)))
    return X,uids,items,uid,iid


def from_matrix_scores(user_batch,scores,items,seen,fallback,topk=20):
    # Caller masks seen items before score ranking.
    out={}
    for bi,u in enumerate(user_batch):
        row=scores[bi];excluded=seen.get(u,set())
        take=min(len(items),max(topk+len(excluded)+8,topk))
        idx=np.argpartition(-row,take-1)[:take] if take<len(items) else np.arange(len(items))
        idx=sorted(idx,key=lambda j:(-float(row[j]),items[j]))
        out[u]=top_unseen(u,(items[j] for j in idx),seen,fallback,topk)
    return out


def matrix_models(fit,users,k,out,profile):
    seen,events,pop=load_interactions(fit)
    if len(pop)<k:raise RuntimeError('Need >=20 catalog items')
    fallback=[i for i,_ in pop.most_common()]
    all_users=sorted(set(seen)|set(users))
    X,uids,items,umap,imap=make_sparse(seen,set(pop),all_users)
    # B: documented substitute, *not* original ALS+NeuMF+SVD group.
    rank=min(32,min(X.shape)-2)
    if rank<1:raise ValueError('Too few users/items for SVD')
    print('Training local SVD proxy for Group B; matrix shape',X.shape, 'rank',rank,flush=True)
    if X.nnz<2:raise ValueError('Too few events')
    # sparse.linalg.svds avoids the dense U×I matrix in RAM.
    try: U,S,Vt=svds(X.astype(np.float32),k=rank,random_state=SEED)
    except Exception as e:
        print('svds failed, retrying with smaller rank:',repr(e));U,S,Vt=svds(X.astype(np.float32),k=max(1,rank//2),random_state=SEED)
    S=S.astype('float32');U=U.astype('float32');Vt=Vt.astype('float32')
    Bpred={}
    bsz=128 if profile!='SMOKE' else 16
    for start in range(0,len(users),bsz):
        ub=users[start:start+bsz]
        ids=[umap[u] for u in ub]
        scores=(U[ids]*S)@Vt
        for ii,u in enumerate(ub):
            for v in seen.get(u,[]):scores[ii,imap[v]]=-np.inf
        Bpred.update(from_matrix_scores(ub,scores,items,seen,fallback,k))

    # C: sparse item co-occurrence weighted by user length, item IDF and shrinkage.
    # This is BM25-like and NOT the teammate's exact BM25-Shrink implementation.
    print('Training BM25-like sparse item model for Group C...',flush=True)
    u_degree=np.asarray(X.sum(axis=1)).ravel()
    avg=float(np.maximum(u_degree.mean(),1.))
    k1,b,shrink,power=1.2,.75,50.,.5
    length_factor=(k1+1)/(k1*(1-b+b*u_degree/avg)+1.)
    Xw=sparse.diags(length_factor.astype(np.float32))@X
    sim=(X.T@Xw).tocsr().astype(np.float32)
    sim.setdiag(0);sim.eliminate_zeros()
    df=np.asarray((X>0).sum(axis=0)).ravel()
    idf=np.maximum(np.log1p((X.shape[0]-df+.5)/(df+.5)),0)
    # Column-aware TF-IDF, shrinkage denominator and popularity dampening.
    for r in range(sim.shape[0]):
        lo,hi=sim.indptr[r],sim.indptr[r+1]
        if lo>=hi:continue
        jj=sim.indices[lo:hi]
        sim.data[lo:hi]=sim.data[lo:hi]*idf[jj]/(shrink+np.power(df[r]*df[jj]+1, power*.5))
    sim.eliminate_zeros()
    # Keep up to K neighbors/row before querying 36K users.
    n_neigh=400 if profile!='SMOKE' else 35
    rowptr=[0]; cols=[];vals=[]
    for i in range(sim.shape[0]):
        sl=slice(sim.indptr[i],sim.indptr[i+1]); d=sim.data[sl];ind=sim.indices[sl]
        if len(d)>n_neigh:
            select=np.argpartition(-d,n_neigh-1)[:n_neigh];d=d[select];ind=ind[select]
        cols.extend(ind.tolist());vals.extend(d.tolist());rowptr.append(len(cols))
    sim=sparse.csr_matrix((np.asarray(vals,dtype='float32'),np.asarray(cols,dtype='int32'),
                          np.asarray(rowptr,dtype='int32')),shape=sim.shape)
    Cpred={}
    pop_array=np.array([pop[i] for i in items],dtype='float32')
    pop_score=np.log1p(pop_array);pop_score/=max(float(np.max(pop_score)),1e-6)
    for start in range(0,len(users),bsz):
        ub=users[start:start+bsz]
        idlist=[umap[u] for u in ub]
        raw=(X[idlist]@sim).toarray().astype('float32')
        # Per-user normalization allows a consistent blend with popularity.
        row_max=np.maximum(raw.max(axis=1,keepdims=True),1e-6)
        raw=.65*(raw/row_max)+.35*pop_score[None,:]
        for ii,u in enumerate(ub):
            for v in seen.get(u,[]):raw[ii,imap[v]]=-np.inf
        Cpred.update(from_matrix_scores(ub,raw,items,seen,fallback,k))
    return Bpred,Cpred


class MaskedSetRecommender(nn.Module):
    def __init__(self, n_items,d=64,layers=2,heads=4,max_len=48,use_order=False):
        super().__init__()
        self.n_items=n_items;self.max_len=max_len;self.use_order=use_order
        self.emb=nn.Embedding(n_items+2,d,padding_idx=0)
        self.pos=nn.Embedding(max_len+1,d) if use_order else None
        enc=nn.TransformerEncoderLayer(d,heads,dim_feedforward=d*2,dropout=.1,batch_first=True)
        self.enc=nn.TransformerEncoder(enc,layers,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(d)
        self.out=nn.Linear(d,n_items+2,bias=False)
        self.out.weight=self.emb.weight
    def forward(self,x):
        y=self.emb(x)*(self.emb.embedding_dim**.5)
        if self.pos is not None:
            positions=torch.arange(x.size(1),device=x.device).view(1,-1)
            y=y+self.pos(positions)
        z=self.enc(y,src_key_padding_mask=(x==0))
        return self.out(self.norm(z[:,0]))


def transformer_predict(fit,users,out,profile,device,seed,use_order=False,topk=20):
    """Mask one item from user's known set. Position-free default; optional timestamp order."""
    seen,events,pop=load_interactions(fit)
    items=sorted(pop)
    idx={it:j+2 for j,it in enumerate(items)}
    rev=['<PAD>','<MASK>']+items
    maxlen=36 if profile=='SMOKE' else 48
    d=32 if profile=='SMOKE' else 64
    ep={'SMOKE':2,'PILOT':6,'FULL':20}[profile]
    bs=32 if profile=='SMOKE' else 256
    stamp=out/'transformer_stamp.json';pt=out/'transformer.pt'
    fp={'fit_sha':sha256(fit),'ep':ep,'d':d,'maxlen':maxlen,'use_order':use_order,'seed':seed}
    model=MaskedSetRecommender(len(items),d=d,max_len=maxlen,use_order=use_order).to(device)
    if stamp.exists() and pt.exists() and json.loads(stamp.read_text())==fp:
        model.load_state_dict(torch.load(pt,map_location=device,weights_only=True))
        print('Transformer checkpoint restored',flush=True)
    else:
        optim=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        pairs=[(u,events[u]) for u in sorted(events) if len(events[u])>=2]
        rng=np.random.default_rng(seed)
        for epoch in range(1,ep+1):
            model.train();rng.shuffle(pairs)
            losses=[]
            for start in range(0,len(pairs),bs):
                batch=pairs[start:start+bs]
                x=np.zeros((len(batch),maxlen),dtype=np.int64)
                y=np.zeros((len(batch),),dtype=np.int64)
                for row,(uid,evs) in enumerate(batch):
                    evs=evs[-(maxlen+8):] if use_order else evs
                    ids=[idx[it] for it,t in evs]
                    picked=int(rng.integers(len(ids)))
                    y[row]=ids[picked]
                    context=[a for n,a in enumerate(ids) if n!=picked]
                    if not use_order:rng.shuffle(context)
                    context=context[-(maxlen-1):]
                    x[row,0]=1;x[row,1:1+len(context)]=context
                x=torch.tensor(x,device=device);y=torch.tensor(y,device=device)
                logits=model(x)
                loss=F.cross_entropy(logits,y)
                optim.zero_grad(set_to_none=True);loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optim.step();losses.append(float(loss.detach().cpu()))
            print(f'Transformer epoch {epoch}/{ep}, loss={np.mean(losses):.4f}',flush=True)
        torch.save(model.state_dict(),pt);atomic_json(stamp,fp)
    model.eval()
    fallback=[i for i,_ in pop.most_common()]
    result={}
    with torch.inference_mode():
        for start in range(0,len(users),bs):
            ub=users[start:start+bs]
            arr=np.zeros((len(ub),maxlen),dtype=np.int64)
            for j,u in enumerate(ub):
                ev=events.get(u,[])
                ids=[idx[i] for i,t in ev]
                if not use_order:ids=sorted(ids)
                ids=ids[-(maxlen-1):]
                arr[j,0]=1;arr[j,1:1+len(ids)]=ids
            logits=model(torch.tensor(arr,device=device)).float().cpu().numpy()
            logits[:,:2]=-np.inf
            for j,u in enumerate(ub):
                for i in seen.get(u,set()):logits[j,idx[i]]=-np.inf
                row=logits[j]
                n=min(len(items),max(topk*3,topk))
                ix=np.argpartition(-row,n-1)[:n]
                ix=sorted(ix,key=lambda a:-row[a])
                result[u]=top_unseen(u,(rev[i] for i in ix),seen,fallback,topk)
    return result


def load_team_adapter(module_path,stage,fit,users,seen,pop,topk,cache_dir):
    """EXTERNAL: module must implement fit_predict(fit_path, users, seen, catalog, topk, stage, cache_dir).
    It must use only `fit_path` interactions to train or load a split-matched checkpoint.
    """
    if not module_path:return None
    if not Path(module_path).is_file():raise FileNotFoundError(f'Team model adapter not found: {module_path}')
    spec=importlib.util.spec_from_file_location('team_external_model_'+stage,str(module_path))
    mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)
    if not hasattr(mod,'fit_predict'):raise ValueError('Team adapter must define fit_predict(...)')
    print(f'Using external team adapter for D on stage {stage}: {module_path}',flush=True)
    res=mod.fit_predict(fit_path=str(fit),users=list(users),seen=seen,catalog=set(pop),
                        topk=topk,stage=stage,cache_dir=str(cache_dir))
    if not isinstance(res,dict):raise TypeError('Adapter must return dict[user_id]=ordered list[item_id]')
    return {str(u):list(map(str,v)) for u,v in res.items()}


def prepare_native(stage,fit,users,root,profile,device,adapter,script_dir,include_transformer=False,transformer_order=False, b_adapter="", c_adapter=""):
    out=root/stage;out.mkdir(parents=True,exist_ok=True)
    seen,events,pop=load_interactions(fit)
    fallback=[i for i,_ in pop.most_common()]
    users=[str(u) for u in users]
    path=out/'target_users.csv';write_users(path,users)
    models={}
    native=native_A_runners(fit,path,out,profile,device,SEED,script_dir)
    # A: exactly the historical ensemble recipe from user's own scripts.
    models['A']=rrf_fuse(native,{'lightgcn':.35,'simplex':.50,'ultragcn':.15},users,seen,fallback,k=60)
    # If team B/C adapters are available, use THEIR native model code, not proxies.
    if b_adapter and c_adapter:
        B=C=None
    else:
        B,C=matrix_models(fit,users,20,out,profile)
    models['B']=load_team_adapter(b_adapter,stage,fit,users,seen,pop,20,out) if b_adapter else B
    models['C']=load_team_adapter(c_adapter,stage,fit,users,seen,pop,20,out) if c_adapter else C
    D=load_team_adapter(adapter,stage,fit,users,seen,pop,20,out)
    if D is not None:models['D']=D
    if include_transformer:
        models['T']=transformer_predict(fit,users,out,profile,device,SEED,use_order=transformer_order)
    for name,p in models.items():
        validate_pred(p,users,seen,set(pop))
    print('Available model groups:',sorted(models),'| stage=',stage,flush=True)
    return models,seen,pop


def make_ranking_features(members,users,seen,pop,fallback,max_users=None,seed=42,truth=None,positive_only=False):
    """No target leakage: 'members' must be fit WITHOUT any targets passed as 'truth'."""
    rng=np.random.default_rng(seed)
    if max_users and len(users)>max_users:
        users=[str(x) for x in sorted(rng.choice(np.array(users),size=max_users,replace=False))]
    names=sorted(members)
    features=[];labels=[];uids=[];itemids=[];groups=[]
    for uid in users:
        # Candidate union from ORIGINAL training-model recommendations only.
        ranks={}
        for name in names:
            for r,it in enumerate(members[name][uid],1):
                ranks.setdefault(it,{})[name]=r
        # Ensure 20 valid candidates. Candidate selection NEVER uses truth.
        if len(ranks)<20:
            for it in fallback:
                if it not in seen.get(uid,set()):ranks.setdefault(it,{})
                if len(ranks)>=20:break
        ranked=sorted(ranks,key=lambda it:(-sum(1/(60+v) for v in ranks[it].values()),it))
        vals=[];ys=[]
        user_history=len(seen.get(uid,set()))
        for item in ranked:
            if item in seen.get(uid,set()):continue
            votes=len(ranks[item])
            invsum=sum(1/(60+v) for v in ranks[item].values())
            v=[math.log1p(user_history),math.log1p(pop.get(item,0)),votes,invsum]
            for name in names:
                rank=ranks[item].get(name,0)
                v.extend([float(rank>0),1/rank if rank else 0.,1/(60+rank) if rank else 0.])
            vals.append(v);ys.append(int(item in truth.get(uid,set())) if truth else 0)
            itemids.append(item);uids.append(uid)
        if positive_only and sum(ys)==0:
            # Drop candidate-only groups with no positive examples from ranker train.
            n=len(vals)
            if n:del itemids[-n:];del uids[-n:]
            continue
        if vals:
            features.extend(vals);labels.extend(ys);groups.append(len(vals))
    X=np.asarray(features,dtype=np.float32)
    y=np.asarray(labels,dtype=np.int8)
    return X,y,np.asarray(uids,dtype=object),np.asarray(itemids,dtype=object),np.asarray(groups,dtype=np.int32),names


def predict_rerank(rank_model,X,uid,item,users,seen,fallback):
    prediction=np.asarray(rank_model.predict(X)).ravel()
    groups=collections.defaultdict(list)
    for i,(u,it) in enumerate(zip(uid,item)):
        groups[str(u)].append((float(prediction[i]),str(it)))
    return {u:top_unseen(u,(v for _,v in sorted(groups[u],key=lambda p:(-p[0],p[1]))),seen,fallback) for u in users}


def fit_rankers(inner_members,outer_members,inner_fit,outer_fit,inner_users,outer_users,inner_truth,outer_truth,out,train_limit=18000,eval_limit=None):
    # Use the same source features in inner and outer models. No shared validation truth in fit.
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    missing=set(inner_members)^set(outer_members)
    if missing:raise RuntimeError(f'Inconsistent inner/outer features: {missing}')
    si,ei,pi=load_interactions(inner_fit)
    so,eo,po=load_interactions(outer_fit)
    inner_fallback=[i for i,_ in pi.most_common()]
    outer_fallback=[i for i,_ in po.most_common()]
    print('Building leak-safe ranker training features...',flush=True)
    Xi,yi,ui,iti,gi,names=make_ranking_features(inner_members,inner_users,si,pi,inner_fallback,
                                                       max_users=train_limit,truth=inner_truth,positive_only=True)
    print('Ranker train rows:',len(yi),'positive=',int(yi.sum()),'groups=',len(gi),flush=True)
    if int(yi.sum())<10 or len(gi)<10:
        print('Too few positives to train a reliable ranker. Skipping Stage 3; expand data/settings.')
        atomic_json(out/'rerank_status.json',{'status':'skipped','reason':'too_few_positive_examples','positives':int(yi.sum())})
        return []
    # Keep exactly the same column order in both stages.
    eval_users=outer_users if eval_limit is None else outer_users[:eval_limit]
    Xe,ye,ue,ite,ge,names2=make_ranking_features(outer_members,eval_users,so,po,outer_fallback,
                                                truth=outer_truth)
    if names!=names2:raise RuntimeError('Ranking model feature schema mismatch')
    rows=[]
    try:
        import lightgbm as lgb
        ranker=lgb.LGBMRanker(n_estimators=180,learning_rate=.05,num_leaves=15,
                             min_child_samples=20,objective='lambdarank',random_state=SEED,
                             verbosity=-1,n_jobs=4)
        ranker.fit(Xi,yi,group=gi)
        p=predict_rerank(ranker,Xe,ue,ite,eval_users,so,outer_fallback)
        result=score_pred(p,{u:outer_truth[u] for u in eval_users})
        row={'model':'LightGBM_Ranker',**result};rows.append(row)
        print('Stage 3',row,flush=True)
        ranker.booster_.save_model(str(out/'lightgbm_ranker.txt'))
    except ImportError:
        print('LightGBM unavailable; on Kaggle enable installation or add library.',flush=True)
    except Exception as e:
        print('LightGBM Ranker failed:',repr(e),flush=True)
        (out/'lightgbm_error.txt').write_text(traceback.format_exc())
    try:
        from catboost import CatBoostRanker
        # All rows within a user group are contiguous, as required by GroupId.
        cb=CatBoostRanker(iterations=120,depth=5,learning_rate=.07,
                          loss_function='PairLogit',random_seed=SEED,verbose=False,
                          thread_count=4)
        cb.fit(Xi,yi,group_id=ui,verbose=False)
        p=predict_rerank(cb,Xe,ue,ite,eval_users,so,outer_fallback)
        result=score_pred(p,{u:outer_truth[u] for u in eval_users})
        row={'model':'CatBoost_Ranker',**result};rows.append(row)
        print('Stage 3',row,flush=True)
        cb.save_model(str(out/'catboost_ranker.cbm'))
    except ImportError:
        print('CatBoost unavailable; on Kaggle enable installation or add library.',flush=True)
    except Exception as e:
        print('CatBoost Ranker failed:',repr(e),flush=True)
        (out/'catboost_error.txt').write_text(traceback.format_exc())
    write_table(out/'reranking_results.csv',rows)
    atomic_json(out/'rerank_status.json',{'status':'completed','training_users':len(gi),
                                         'training_rows':len(yi),'training_positive':int(yi.sum()),
                                         'feature_groups':names,'models':[r['model'] for r in rows],
                                         'evaluation_users':len(eval_users)})
    return rows


def getargs():
    ap=argparse.ArgumentParser()
    ap.add_argument('--shared-zip',default='')
    ap.add_argument('--input-root',default='/kaggle/input')
    ap.add_argument('--work',default='/kaggle/working/team_allsteps')
    ap.add_argument('--scripts',default='/kaggle/working')
    ap.add_argument('--profile',choices=['SMOKE','PILOT','FULL'],default='PILOT')
    ap.add_argument('--team-b-adapter',default='',help='Teammate B Python model adapter with fit_predict(...)')
    ap.add_argument('--team-c-adapter',default='',help='Teammate C Python model adapter with fit_predict(...)')
    ap.add_argument('--team-d-adapter',default='',help='Teammate D Python model adapter with fit_predict(...)')
    ap.add_argument('--require-all-real-groups',action='store_true',help='Fail instead of using B/C proxies or omitting D')
    ap.add_argument('--time-order',action='store_true',help='Optional position-aware masked Transformer (timestamp sorted), not strict next-item SASRec')
    ap.add_argument('--device',default='cuda:0' if torch.cuda.is_available() else 'cpu')
    ap.add_argument('--ranker-train-users',type=int,default=18000)
    ap.add_argument('--ranker-eval-users',type=int,default=0,help='0=all common validation users')
    return ap.parse_args()


def main(args):
    work=Path(args.work);work.mkdir(parents=True,exist_ok=True)
    print('GPU:',args.device,'Profile:',args.profile,flush=True)
    if args.require_all_real_groups and not (args.team_b_adapter and args.team_c_adapter and args.team_d_adapter):
        raise RuntimeError('Actual 4-team reproduction requires B/C/D fit_predict adapters. Current B/C are proxies and D is skipped.')
    teamkit,valid,manifest=locate_teamkit(args.shared_zip,args.input_root,work)
    outer_fit=valid/'val_train.csv';outer_truth_path=valid/'val_truth.csv';outer_users_path=valid/'val_users.csv'
    outer_truth=teamkit.read_truth(outer_truth_path)
    outer_users=teamkit.read_users(outer_users_path)
    nested=work/'nested_split';nested.mkdir(exist_ok=True)
    inner_fit=nested/'inner_train.csv';inner_truth_path=nested/'inner_truth.csv';inner_users_path=nested/'inner_users.csv'
    inner_split(outer_fit,inner_fit,inner_truth_path,inner_users_path)
    inner_truth=teamkit.read_truth(inner_truth_path)
    inner_users=teamkit.read_users(inner_users_path)
    if not outer_users:raise RuntimeError('No users')
    print('\n===== STAGE 1 / TEAM MODEL ENSEMBLE =====',flush=True)
    outer,seen,pop=prepare_native('outer',outer_fit,outer_users,work,args.profile,args.device,
                                    args.team_d_adapter,Path(args.scripts),False,args.time_order,
                                    args.team_b_adapter,args.team_c_adapter)
    outer_diag=work/'step1';outer_diag.mkdir(exist_ok=True)
    all_scores(outer_users,seen,pop,[x for x,_ in pop.most_common()],outer,outer_truth,outer_diag)
    complementarity(outer,outer_truth,outer_diag)
    ens1,p1=group_ABLATION(outer,outer_users,seen,pop,outer_truth,outer_diag)
    print('Stage1 winner:',ens1[0]['model'] if ens1 else 'No eligible combinations',flush=True)
    print('\n===== STAGE 2 / MASKED-ITEM TRANSFORMER =====',flush=True)
    T=transformer_predict(outer_fit,outer_users,work/'outer',args.profile,args.device,SEED,
                          use_order=args.time_order)
    validate_pred(T,outer_users,seen,set(pop))
    outer['T']=T
    transformer_metric=score_pred(T,outer_truth)
    print('Transformer:',transformer_metric,flush=True)
    atomic_json(work/'step2_transformer.json',transformer_metric)
    ens2,p2=group_ABLATION(outer,outer_users,seen,pop,outer_truth,outer_diag,extra_T=True)
    complementarity(outer,outer_truth,outer_diag)
    print('Stage2 winner:',ens2[0]['model'] if ens2 else 'None',flush=True)
    print('\n===== STAGE 3 / LEAK-SAFE RERANKING =====',flush=True)
    inner,inner_seen,inner_pop=prepare_native('inner',inner_fit,inner_users,work,args.profile,args.device,
                                               args.team_d_adapter,Path(args.scripts),True,args.time_order,
                                               args.team_b_adapter,args.team_c_adapter)
    step3=work/'step3';step3.mkdir(parents=True,exist_ok=True)
    ranker_rows=fit_rankers(inner,outer,inner_fit,outer_fit,inner_users,outer_users,inner_truth,outer_truth,
                            step3,
                            train_limit=args.ranker_train_users,
                            eval_limit=args.ranker_eval_users or None)
    summary={'split_id':manifest.get('split_id'),'profile':args.profile,
             'stage1_best':ens1[0] if ens1 else None,
             'stage2_transformer':transformer_metric,
             'stage2_best':ens2[0] if ens2 else None,
             'stage3_results':ranker_rows,
             'external_team_adapters_connected':{key:bool(getattr(args,f'team_{key.lower()}_adapter')) for key in ('B','C','D')},
             'group_provenance':{
                'A':'Own original LightGCN + SimpleX + UltraGCN runner implementations',
                'B':('teammate real adapter' if args.team_b_adapter else 'PROXY: sparse truncated SVD only; NOT ALS+NeuMF+SVD'),
                'C':('teammate real adapter' if args.team_c_adapter else 'PROXY: Popularity + BM25-like sparse co-occurrence; NOT exact team BM25-Shrink'),
                'D':('teammate real adapter' if args.team_d_adapter else 'ABSENT: MiniLM+CLIP+VBPR unavailable'),
                'T':'New from-scratch masked-item Transformer',
             },
             'IMPORTANT':'Validation-only. Missing team adapters mean this is not a faithful four-person ensemble. No Kaggle Test submission.'}
    atomic_json(work/'summary.json',summary)
    print('\n=== ALL 3 STAGES COMPLETE ===',json.dumps(summary,ensure_ascii=False,indent=2),flush=True)
    # Safe to zip small reports only. No private validation/training data inside.
    zpath=work.parent/'team_allsteps_results.zip'
    with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
        for p in [work/'summary.json',work/'step2_transformer.json',
                  *list((work/'step1').glob('*.csv')),*list((work/'step1').glob('*.json')),
                  *list((work/'step3').glob('*.csv')),*list((work/'step3').glob('*.json'))]:
            if p.exists():z.write(p,arcname=p.relative_to(work))
    print('Download result ZIP:',zpath,flush=True)

if __name__=='__main__':
    args=getargs();main(args)


In [ ]:
%%writefile /kaggle/working/team_allsteps_REAL_ablation.py
#!/usr/bin/env python3
"""Amazon Top-K: actual multi-config Stage 1/2/3 ablation, validation only.

Group A: original scripts (LightGCN, SimpleX, UltraGCN) trained anew.
Group B/C: PROXIES until team adapters supplied; D: absent until supplied.
Native original fixed engine supplies data validation, model runners and ranks.
No test.csv, no kaggle submission, and no val_truth leakage into ranker fit.
"""
from __future__ import annotations
import argparse, collections, gc, json, math, random, sys, time, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd

import team_allsteps_model_pipeline as core

SEED=42

def jwrite(path,obj):
    core.atomic_json(path,obj)

def configs(profile):
    if profile=='SMOKE':
        return [{'name':'T_smoke_d32_L1','d':32,'layers':1,'heads':4,'epochs':2,'lr':.001,'batch':8,'maxlen':24,'eval_every':1}]
    if profile=='BALANCED':
        return [
           {'name':'T0_d64_L2_lr001','d':64,'layers':2,'heads':4,'epochs':40,'lr':.001,'batch':256,'maxlen':48,'eval_every':10},
           {'name':'T1_d128_L2_lr001','d':128,'layers':2,'heads':4,'epochs':60,'lr':.001,'batch':256,'maxlen':48,'eval_every':10},
           {'name':'T2_d64_L3_lr0005','d':64,'layers':3,'heads':4,'epochs':80,'lr':.0005,'batch':256,'maxlen':48,'eval_every':10},
        ]
    if profile=='DEEP':
        return [
           {'name':'T0_d64_L2_lr001','d':64,'layers':2,'heads':4,'epochs':70,'lr':.001,'batch':256,'maxlen':48,'eval_every':10},
           {'name':'T1_d128_L2_lr001','d':128,'layers':2,'heads':4,'epochs':90,'lr':.001,'batch':256,'maxlen':48,'eval_every':10},
           {'name':'T2_d64_L3_lr0005','d':64,'layers':3,'heads':4,'epochs':110,'lr':.0005,'batch':256,'maxlen':48,'eval_every':10},
           {'name':'T3_d128_L3_lr0005','d':128,'layers':3,'heads':4,'epochs':110,'lr':.0005,'batch':192,'maxlen':48,'eval_every':10},
        ]
    raise ValueError(profile)

def make_group_weights(available, include_t=False):
    base=[
       ('Aonly',{'A':1}),('Bonly',{'B':1}),('Conly',{'C':1}),('Donly',{'D':1}),
       ('AC5050',{'A':.5,'C':.5}),('AC7030',{'A':.7,'C':.3}),('AC3070',{'A':.3,'C':.7}),
       ('AB6040',{'A':.6,'B':.4}),('BC5050',{'B':.5,'C':.5}),
       ('ABC_502030',{'A':.5,'B':.2,'C':.3}),
       ('ABC_402040',{'A':.4,'B':.2,'C':.4}),
       ('ABC_301060',{'A':.3,'B':.1,'C':.6}),
       ('ABC_601030',{'A':.6,'B':.1,'C':.3}),
       ('DA6040',{'D':.6,'A':.4}),('DC6040',{'D':.6,'C':.4}),
       ('DAC_502525',{'D':.5,'A':.25,'C':.25}),
       ('DAC_602020',{'D':.6,'A':.2,'C':.2}),
       ('DABC_45252010',{'D':.45,'A':.25,'C':.2,'B':.1}),
       ('DABC_equal',{'D':.25,'A':.25,'C':.25,'B':.25}),
    ]
    if include_t:
        base += [
          ('Tonly',{'T':1}),('AT_8020',{'A':.8,'T':.2}),('AT_6040',{'A':.6,'T':.4}),
          ('CT_8020',{'C':.8,'T':.2}),('CT_6040',{'C':.6,'T':.4}),
          ('ACT_402020',{'A':.4,'C':.4,'T':.2}),('ACT_303040',{'A':.3,'C':.3,'T':.4}),
          ('ABCT_40202020',{'A':.4,'B':.2,'C':.2,'T':.2}),
          ('DTACT_503020',{'D':.5,'A':.3,'T':.2}),
        ]
    return [(name,w) for name,w in base if set(w).issubset(available)]

def run_fusions(members,users,seen,pop,truth,output,profile,stage):
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    available=set(members)
    combos=make_group_weights(available,include_t='T' in available)
    fallback=[i for i,_ in pop.most_common()]
    k_values=[20,60] if profile=='SMOKE' else [10,20,60]
    rows=[]
    print(f'{stage}: {len(combos)} candidate recipes x {len(k_values)} RRF k values = {len(combos)*len(k_values)} runs',flush=True)
    for ix,(nm,weights) in enumerate(combos,1):
        for k in k_values:
            pred=core.rrf_fuse(members,weights,users,seen,fallback,k=k)
            sc=core.score_pred(pred,truth)
            rows.append({'experiment':nm+f'_k{k}','weights':json.dumps(weights,sort_keys=True),
                         'rrf_k':k,'stage':stage,**sc})
        if ix%4==0:print(f'  {stage} {ix}/{len(combos)} recipes',flush=True)
    rows.sort(key=lambda r:(-r['recall_at_20'],r['experiment']))
    core.write_table(output/f'{stage}_leaderboard.csv',rows)
    jwrite(output/f'{stage}_best.json',rows[0] if rows else {})
    return rows

def predict_transformer(model,events,idx,rev,users,seen,pop,device,maxlen,batch=256,topk=20):
    model.eval(); fallback=[i for i,_ in pop.most_common()];pred={}
    n_items=len(rev)-2
    with torch.inference_mode():
        for start in range(0,len(users),batch):
            ub=users[start:start+batch]
            x=np.zeros((len(ub),maxlen),np.int64)
            for j,u in enumerate(ub):
                context=sorted(idx[i] for i,t in events.get(u,[]))[-(maxlen-1):]
                x[j,0]=1;x[j,1:1+len(context)]=context
            logits=model(torch.from_numpy(x).to(device)).float().cpu().numpy()
            logits[:,:2]=-np.inf
            for j,u in enumerate(ub):
                for it in seen.get(u,set()):logits[j,idx[it]]=-np.inf
                row=logits[j]
                take=min(n_items,max(topk*3,topk))
                keep=np.argpartition(-row,take-1)[:take]
                keep=sorted(keep,key=lambda a:(-row[a],a))
                pred[u]=core.top_unseen(u,(rev[p] for p in keep),seen,fallback,topk)
    return pred

def train_transformer(fit,users,truth,output,config,device,select_best=True):
    """True multi-epoch masked-set recommender; no positional timestamps required.

    SELECT BEST is enabled only on outer common validation for ablations.
    Inner ranker training uses fixed best outer epoch; inner labels NEVER consulted.
    """
    out=Path(output)/config['name'];out.mkdir(parents=True,exist_ok=True)
    seen,events,pop=core.load_interactions(fit)
    users=[str(u) for u in users]
    items=sorted(pop);idx={v:i+2 for i,v in enumerate(items)};rev=['<PAD>','<MASK>']+items
    cfg={k:v for k,v in config.items() if k!='best_epoch'}
    stamp={'fit_sha256':core.sha256(fit),'cfg':cfg,'select_best':select_best,
           'device_type':device.split(':')[0],'script_version':'ablation-v2'}
    ck=out/'best_model.pt';st=out/'stamp.json';predfile=out/'predictions.csv';met=out/'metrics.json'
    if ck.exists() and predfile.exists() and st.exists() and json.loads(st.read_text())==stamp:
        pr=core.read_pred(predfile)
        core.validate_pred(pr,users,seen,set(pop))
        result=json.loads(met.read_text())
        print('Transformer cached:',config['name'],result.get('best_epoch'),flush=True)
        return pr,result
    torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
    d=config['d']; maxlen=config['maxlen'];batch=config['batch'];epochs=config['epochs']
    model=core.MaskedSetRecommender(len(items),d=d,layers=config['layers'],heads=config['heads'],max_len=maxlen,use_order=False).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=config['lr'],weight_decay=.01)
    pairs=[(u,events[u]) for u in sorted(events) if len(events[u])>=2]
    if not pairs:raise RuntimeError('Not enough training pairs')
    rg=np.random.default_rng(SEED)
    best=-1.;best_epoch=0;best_pred=None;hist=[]
    for epoch in range(1,epochs+1):
        model.train();rg.shuffle(pairs); losses=[]
        for lo in range(0,len(pairs),batch):
            b=pairs[lo:lo+batch]
            x=np.zeros((len(b),maxlen),np.int64);y=np.zeros(len(b),np.int64)
            for j,(u,ev) in enumerate(b):
                ids=[idx[it] for it,t in ev]
                target=int(rg.integers(len(ids)))
                y[j]=ids[target]
                ctx=ids[:target]+ids[target+1:]
                rg.shuffle(ctx);ctx=ctx[:maxlen-1]
                x[j,0]=1;x[j,1:len(ctx)+1]=ctx
            logits=model(torch.from_numpy(x).to(device))
            loss=F.cross_entropy(logits,torch.from_numpy(y).to(device))
            opt.zero_grad(set_to_none=True);loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            opt.step();losses.append(float(loss.detach().cpu()))
        if epoch==1 or epoch%5==0 or epoch==epochs:
            print(f'  {config["name"]}: epoch {epoch}/{epochs}, loss {np.mean(losses):.4f}',flush=True)
        evaluation=(epoch==epochs or epoch%config['eval_every']==0)
        if evaluation:
            if select_best:
                pred=predict_transformer(model,events,idx,rev,users,seen,pop,device,maxlen,batch,20)
                score=core.score_pred(pred,truth)
                row={'epoch':epoch,'loss':float(np.mean(losses)),**score};hist.append(row)
                print(f'  {config["name"]}: val at {epoch} = {score["recall_at_20"]:.6f}',flush=True)
                if score['recall_at_20']>best:
                    best=score['recall_at_20'];best_epoch=epoch;best_pred=pred
                    torch.save(model.state_dict(),ck)
            else:
                if epoch==epochs:
                    best_epoch=epoch
                    torch.save(model.state_dict(),ck)
    if select_best:
        assert best_pred is not None
        pred=best_pred
        result={'experiment':config['name'],'best_epoch':best_epoch,'recall_at_20':best,'evaluations':len(hist), 'config':config}
        jwrite(out/'history.json',hist)
    else:
        pred=predict_transformer(model,events,idx,rev,users,seen,pop,device,maxlen,batch,20)
        result={'experiment':config['name'],'best_epoch':best_epoch,'inner_stage_no_validation_selection':True,'config':config}
    core.validate_pred(pred,users,seen,set(pop)); core.write_pred(predfile,pred,users)
    jwrite(met,result);jwrite(st,stamp)
    del model;gc.collect()
    if device.startswith('cuda'):torch.cuda.empty_cache()
    return pred,result

def fit_ranker_ablation(inner_members,outer_members,inner_fit,outer_fit,inner_users,outer_users,inner_truth,outer_truth,out,profile,train_limit=18000):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    ni=set(inner_members);no=set(outer_members)
    if ni!=no:raise RuntimeError(f'Reranker feature model names differ: {ni} vs {no}')
    si,_,pi=core.load_interactions(inner_fit)
    so,_,po=core.load_interactions(outer_fit)
    fb_i=[i for i,_ in pi.most_common()];fb_o=[i for i,_ in po.most_common()]
    Xi,yi,ui,iti,gi,names=core.make_ranking_features(inner_members,inner_users,si,pi,fb_i,max_users=train_limit,truth=inner_truth,positive_only=True)
    Xe,ye,ue,ite,ge,names2=core.make_ranking_features(outer_members,outer_users,so,po,fb_o,truth=outer_truth)
    if names!=names2:raise RuntimeError('Feature order mismatch')
    if len(gi)<10 or yi.sum()<10:raise RuntimeError(f'Not enough positive training instances: {yi.sum()}')
    print(f'Stage3: {len(Xi)} training rows; positives={int(yi.sum())}; user groups={len(gi)}; outer candidates={len(Xe)}',flush=True)
    models=[
      ('LGBM_small', 'lgb', {'n_estimators':200,'learning_rate':.05,'num_leaves':15,'min_child_samples':20}),
      ('LGBM_medium', 'lgb', {'n_estimators':400,'learning_rate':.03,'num_leaves':31,'min_child_samples':30}),
      ('LGBM_regularized','lgb',{'n_estimators':350,'learning_rate':.04,'num_leaves':15,'min_child_samples':50}),
      ('CAT_depth4','cat',{'iterations':250,'depth':4,'learning_rate':.05,'l2_leaf_reg':3}),
      ('CAT_depth6','cat',{'iterations':400,'depth':6,'learning_rate':.035,'l2_leaf_reg':6}),
    ]
    if profile=='SMOKE':models=[('LGBM_smoke','lgb',{'n_estimators':10,'learning_rate':.1,'num_leaves':7,'min_child_samples':2})]
    if profile=='DEEP':models += [('CAT_depth5','cat',{'iterations':500,'depth':5,'learning_rate':.03,'l2_leaf_reg':3})]
    rows=[]
    for name,typ,cfg in models:
        print('  Training ranker:',name,cfg,flush=True)
        try:
            if typ=='lgb':
                import lightgbm as lgb
                m=lgb.LGBMRanker(objective='lambdarank',random_state=SEED,verbosity=-1,n_jobs=4,**cfg)
                m.fit(Xi,yi,group=gi)
            else:
                from catboost import CatBoostRanker
                m=CatBoostRanker(loss_function='PairLogit',random_seed=SEED,verbose=False,thread_count=4,**cfg)
                m.fit(Xi,yi,group_id=ui,verbose=False)
            pr=core.predict_rerank(m,Xe,ue,ite,outer_users,so,fb_o)
            sc=core.score_pred(pr,outer_truth)
            rows.append({'experiment':name,'type':typ,'params':json.dumps(cfg),**sc})
            print('  ',name,'validation Recall@20',sc['recall_at_20'],flush=True)
            if typ=='lgb':m.booster_.save_model(str(out/(name+'.txt')))
            else:m.save_model(str(out/(name+'.cbm')))
        except ImportError as ex:
            print('Skipping unavailable ranker dependency',name,ex,flush=True)
        del m
    rows.sort(key=lambda x:-x['recall_at_20'])
    core.write_table(out/'stage3_ranker_ablation.csv',rows)
    jwrite(out/'stage3_best.json',rows[0] if rows else {})
    return rows

def run(args):
    work=Path(args.work);work.mkdir(parents=True,exist_ok=True)
    module_root=Path(args.scripts)
    tk,val,manifest=core.locate_teamkit(args.shared_zip,Path(args.input_root),work)
    valid=val/'val_train.csv';truth=tk.read_truth(val/'val_truth.csv');users=tk.read_users(val/'val_users.csv')
    print('== Stage 1: FULL trained base models (not PILOT) ==',flush=True)
    members,seen,pop=core.prepare_native('outer',valid,users,work,'SMOKE' if args.profile=='SMOKE' else 'FULL',
        args.device,args.d_adapter,module_root,False,False,args.b_adapter,args.c_adapter)
    core.all_scores(users,seen,pop,[i for i,_ in pop.most_common()],members,truth,work/'stage1')
    core.complementarity(members,truth,work/'stage1')
    s1=run_fusions(members,users,seen,pop,truth,work/'stage1',args.profile,'stage1')
    print('Stage1 best:',s1[0] if s1 else None,flush=True)
    print('== Stage 2: Transformer multi-config ablation ==',flush=True)
    variants=configs(args.profile);rows=[];t_preds={};best=None
    for cfg in variants:
        pr,r=train_transformer(valid,users,truth,work/'stage2'/'transformers',cfg,args.device,True)
        sc=core.score_pred(pr,truth)
        rows.append({'experiment':cfg['name'],'best_epoch':r['best_epoch'],'configs':json.dumps(cfg),**sc})
        t_preds[cfg['name']]=pr
        if best is None or sc['recall_at_20']>best[0]:best=(sc['recall_at_20'],cfg,r)
    rows.sort(key=lambda r:-r['recall_at_20'])
    core.write_table(work/'stage2'/'transformer_ablation.csv',rows)
    best_cf=dict(best[1]);best_cf['best_epoch']=int(best[2]['best_epoch'])
    jwrite(work/'stage2'/'selected_transformer.json',best_cf)
    members['T']=t_preds[best_cf['name']]
    core.complementarity(members,truth,work/'stage2')
    s2=run_fusions(members,users,seen,pop,truth,work/'stage2',args.profile,'stage2')
    print('Stage2 best:',s2[0] if s2 else None,flush=True)
    print('== Stage 3: INNER-split leak-safe ranker multi-config ablation ==',flush=True)
    nested=work/'inner_split';nested.mkdir(exist_ok=True,parents=True)
    inner_fit=nested/'train.csv';inner_truth_file=nested/'truth.csv';inner_users_file=nested/'users.csv'
    core.inner_split(valid,inner_fit,inner_truth_file,inner_users_file)
    truth_i=tk.read_truth(inner_truth_file);users_i=tk.read_users(inner_users_file)
    inner,si,pi=core.prepare_native('inner',inner_fit,users_i,work,'SMOKE' if args.profile=='SMOKE' else 'FULL',
        args.device,args.d_adapter,module_root,False,False,args.b_adapter,args.c_adapter)
    # Use SAME selected model hyperparameters on inner split; fixed epoch from outer,
    # without looking at INNER labels. The inner train labels are used ONLY by rankers.
    inner_cfg=dict(best_cf);inner_cfg['epochs']=int(best_cf['best_epoch'])
    inner_T,_=train_transformer(inner_fit,users_i,truth_i,work/'stage3'/'inner_transformer',inner_cfg,args.device,False)
    inner['T']=inner_T
    s3=fit_ranker_ablation(inner,members,inner_fit,valid,users_i,users,truth_i,truth,work/'stage3',args.profile,args.ranker_train_users)
    status={'split_id':manifest['split_id'],'profile':args.profile,
            'stage1_best':s1[0] if s1 else None,'transformer_models':rows,
            'stage2_best':s2[0] if s2 else None,'stage3_rankers':s3,
            'actual_team_adapters':{k:bool(v) for k,v in [('B',args.b_adapter),('C',args.c_adapter),('D',args.d_adapter)]},
            'provenance_note':'B/C are surrogate models and D absent without adapters. This is NOT real 4-team ensemble.',
            'no_submission':True, 'selection_bias_note':'Repeated outer validation tuning can overfit; consider a final untouched holdout.'}
    jwrite(work/'summary.json',status)
    archive=work.parent/'team_real_ablation_results.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in work.rglob('*'):
            if p.is_file() and p.suffix in ('.csv','.json') and 'predictions' not in p.name and 'teamkit_unpack' not in p.parts and 'inner_split' not in p.parts and not p.name.startswith('target_users'):
                if p.stat().st_size<5_000_000:z.write(p,arcname=p.relative_to(work))
    print('RESULTS ARCHIVE:',archive,flush=True)
    print(json.dumps({'stage1_best':status['stage1_best'],'stage2_best':status['stage2_best'],'stage3_best':s3[0] if s3 else None},ensure_ascii=False,indent=2),flush=True)

def parse():
    p=argparse.ArgumentParser()
    p.add_argument('--input-root',default='/kaggle/input');p.add_argument('--shared-zip',default='')
    p.add_argument('--work',default='/kaggle/working/team_real_ablation');p.add_argument('--scripts',default='/kaggle/working')
    p.add_argument('--profile',choices=['SMOKE','BALANCED','DEEP'],default='BALANCED')
    p.add_argument('--device',default='cuda:0' if torch.cuda.is_available() else 'cpu')
    p.add_argument('--ranker-train-users',type=int,default=18000)
    p.add_argument('--b-adapter',default='');p.add_argument('--c-adapter',default='');p.add_argument('--d-adapter',default='')
    p.add_argument('--require-real-team',action='store_true')
    a=p.parse_args()
    if a.require_real_team and not all((a.b_adapter,a.c_adapter,a.d_adapter)):
        raise ValueError('Actual team model ablation requires all B/C/D adapters; current proxies not valid replacements.')
    return a

if __name__=='__main__':run(parse())


In [ ]:
%%writefile /kaggle/working/teamkit.py
"""Shared, standard-library-only evaluation utilities for the Amazon Top-20 team.

All interactions are targets, regardless of rating. IDs are kept as strings so
CSV round-trips do not change them. The public CSV contract is user_id,item_id.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Iterable, Mapping, Sequence


TOP_K = 20
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20


@contextmanager
def open_source(path: str | None, competition_zip: str | None, member: str):
    """Open a UTF-8 file, either directly or as a named competition ZIP member."""
    if (path is None) == (competition_zip is None):
        raise ValueError("Provide exactly one of a file path or --competition-zip")
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive:
            with archive.open(member) as raw:
                with io.TextIOWrapper(raw, encoding="utf-8-sig", newline="") as stream:
                    yield stream
    else:
        with open(path, "r", encoding="utf-8-sig", newline="") as stream:
            yield stream


def _reader(stream, required: Sequence[str]) -> csv.DictReader:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
        raise ValueError(f"Required columns: {', '.join(required)}")
    return reader


def _clean_id(value: str | None, name: str) -> str:
    if value is None or value == "" or value != value.strip() or any(c.isspace() for c in value):
        raise ValueError(f"Invalid {name}: {value!r}")
    return value


def _stable_rank(seed: int, user_id: str, item_id: str) -> bytes:
    payload = f"{seed}\0{user_id}\0{item_id}".encode("utf-8")
    return hashlib.blake2b(payload, digest_size=16).digest()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _source_sha256(path: str | None, competition_zip: str | None, member: str) -> str:
    digest = hashlib.sha256()
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive, archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    else:
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def make_split(
    train_path: str | None,
    competition_zip: str | None,
    output_dir: str | Path,
    seed: int = SPLIT_SEED,
    fraction: float = HOLDOUT_FRACTION,
    overwrite: bool = False,
) -> dict:
    """Make a stable, per-user split on unique user-item pairs.

    Hold out ceil(fraction * interactions), at most 20, leaving at least one
    interaction in fit. The hash rank is independent of input row order.
    """
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    names = ("val_train.csv", "val_truth.csv", "val_users.csv", "manifest.json")
    existing = [name for name in names if (output_dir / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Existing split files: {existing}; use --overwrite to replace")

    with open_source(train_path, competition_zip, "train.csv") as stream:
        reader = _reader(stream, ("user_id", "item_id"))
        columns = list(reader.fieldnames or [])
        rows: list[tuple[str, ...]] = []
        user_items: dict[str, list[str]] = defaultdict(list)
        pairs: set[tuple[str, str]] = set()
        for row in reader:
            if None in row:
                raise ValueError("Malformed train.csv row with extra fields")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            pair = (user_id, item_id)
            if pair in pairs:
                raise ValueError(f"Duplicate interaction: {pair}")
            pairs.add(pair)
            user_items[user_id].append(item_id)
            rows.append(tuple(row[column] for column in columns))

    held: set[tuple[str, str]] = set()
    for user_id, items in user_items.items():
        if len(items) < 2:
            continue
        n_held = min(TOP_K, len(items) - 1, max(1, math.ceil(fraction * len(items))))
        chosen = sorted(items, key=lambda item: (_stable_rank(seed, user_id, item), item))[:n_held]
        held.update((user_id, item) for item in chosen)

    fit_path = output_dir / "val_train.csv"
    truth_path = output_dir / "val_truth.csv"
    users_path = output_dir / "val_users.csv"
    user_col, item_col = columns.index("user_id"), columns.index("item_id")
    with fit_path.open("w", encoding="utf-8", newline="") as fit_stream, truth_path.open(
        "w", encoding="utf-8", newline=""
    ) as truth_stream:
        fit_writer, truth_writer = csv.writer(fit_stream), csv.writer(truth_stream)
        fit_writer.writerow(columns)
        truth_writer.writerow(("user_id", "item_id"))
        for row in rows:
            pair = (row[user_col], row[item_col])
            if pair in held:
                truth_writer.writerow(pair)
            else:
                fit_writer.writerow(row)

    with users_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id",))
        for user_id in user_items:
            if len(user_items[user_id]) >= 2:
                writer.writerow((user_id,))

    source_hash = _source_sha256(train_path, competition_zip, "train.csv")
    split_key = f"{source_hash}|blake2b16|{seed}|{fraction}|ceil20keep1"
    manifest = {
        "split_id": hashlib.sha256(split_key.encode()).hexdigest()[:16],
        "algorithm": "per-user blake2b-128 rank over seed, user_id, item_id",
        "seed": seed,
        "holdout_fraction": fraction,
        "holdout_count": "min(20, n-1, max(1, ceil(fraction*n))) for n>=2",
        "source_train_sha256": source_hash,
        "source_rows": len(rows),
        "source_users": len(user_items),
        "fit_rows": len(rows) - len(held),
        "truth_rows": len(held),
        "validation_users": sum(len(items) >= 2 for items in user_items.values()),
        "files_sha256": {name: _sha256(output_dir / name) for name in names[:3]},
    }
    with (output_dir / "manifest.json").open("w", encoding="utf-8") as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    return manifest


def verify_split(output_dir: str | Path) -> dict:
    output_dir = Path(output_dir)
    with (output_dir / "manifest.json").open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    for name, expected in manifest["files_sha256"].items():
        actual = _sha256(output_dir / name)
        if actual != expected:
            raise ValueError(f"Split file checksum mismatch: {name}")
    return manifest


def read_users(path: str | Path) -> list[str]:
    users: list[str] = []
    seen: set[str] = set()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id",)):
            if None in row:
                raise ValueError(f"Malformed user CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in seen:
                raise ValueError(f"Duplicate user in {path}: {user_id}")
            seen.add(user_id)
            users.append(user_id)
    return users


def read_interactions(path: str | Path) -> tuple[dict[str, set[str]], Counter[str]]:
    """Return seen items by user and item popularity from a fit CSV."""
    seen: dict[str, set[str]] = defaultdict(set)
    popularity: Counter[str] = Counter()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed interaction CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id not in seen[user_id]:
                seen[user_id].add(item_id)
                popularity[item_id] += 1
    return dict(seen), popularity


def read_truth(path: str | Path) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed truth CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id in truth[user_id]:
                raise ValueError(f"Duplicate truth item for user {user_id}: {item_id}")
            truth[user_id].add(item_id)
    return dict(truth)


def read_predictions(path: str | Path) -> dict[str, list[str]]:
    predictions: dict[str, list[str]] = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["user_id", "item_id"]:
            raise ValueError("Prediction columns must be exactly user_id,item_id")
        for row in reader:
            if None in row:
                raise ValueError("Malformed prediction CSV row")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in predictions:
                raise ValueError(f"Duplicate prediction user: {user_id}")
            value = row["item_id"]
            if value is None:
                raise ValueError(f"Missing item list for user {user_id}")
            items = value.split(" ")
            if len(items) != TOP_K or any(not item or item != item.strip() for item in items):
                raise ValueError(f"User {user_id} needs exactly {TOP_K} space-separated item IDs")
            if len(set(items)) != TOP_K:
                raise ValueError(f"Duplicate recommended item for user {user_id}")
            predictions[user_id] = items
    return predictions


def validate_predictions(
    predictions: Mapping[str, Sequence[str]],
    expected_users: Iterable[str],
    seen: Mapping[str, set[str]],
    catalog: set[str],
) -> None:
    expected = set(expected_users)
    actual = set(predictions)
    if actual != expected:
        missing, extra = expected - actual, actual - expected
        raise ValueError(f"User mismatch: {len(missing)} missing, {len(extra)} extra")
    for user_id, items in predictions.items():
        if len(items) != TOP_K or len(set(items)) != TOP_K:
            raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
        unknown = set(items) - catalog
        if unknown:
            raise ValueError(f"Unknown item for user {user_id}: {sorted(unknown)[:3]}")
        repeated = set(items) & seen.get(user_id, set())
        if repeated:
            raise ValueError(f"Seen item recommended to user {user_id}: {sorted(repeated)[:3]}")


def recall_at_20(predictions: Mapping[str, Sequence[str]], truth: Mapping[str, set[str]]) -> dict:
    """Competition-style pooled Recall@20, not mean of user-level recalls."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user_id]) for user_id, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if denominator == 0:
        raise ValueError("No validation truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def top20_unseen(
    user_id: str,
    ranked_candidates: Iterable[str],
    seen: Mapping[str, set[str]],
    fallback_items: Iterable[str],
    catalog: set[str],
) -> list[str]:
    """Filter a model ranking; fill short lists with popular unseen items."""
    chosen: list[str] = []
    used = set(seen.get(user_id, set()))
    for source in (ranked_candidates, fallback_items):
        for item_id in source:
            item_id = str(item_id)
            if item_id not in catalog or item_id in used:
                continue
            chosen.append(item_id)
            used.add(item_id)
            if len(chosen) == TOP_K:
                return chosen
    raise ValueError(f"Fewer than {TOP_K} available unseen items for user {user_id}")


def write_predictions(path: str | Path, users: Iterable[str], predictions: Mapping[str, Sequence[str]]) -> None:
    """Write one row per user in the provided order, using single spaces."""
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user_id in users:
            items = predictions[user_id]
            if len(items) != TOP_K or len(set(items)) != TOP_K:
                raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
            writer.writerow((user_id, " ".join(map(str, items))))


def filter_reviews(
    fit_path: str | Path,
    output_path: str | Path,
    reviews_path: str | None = None,
    competition_zip: str | None = None,
) -> dict:
    """Keep only training-pair reviews to prevent validation target leakage."""
    seen, _ = read_interactions(fit_path)
    allowed = {(user_id, item_id) for user_id, items in seen.items() for item_id in items}
    found: set[tuple[str, str]] = set()
    written = 0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_path = tempfile.mkstemp(prefix=".reviews-", suffix=".tmp", dir=output_path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as output, open_source(
            reviews_path, competition_zip, "train_reviews.jsonl"
        ) as source:
            for line in source:
                review = json.loads(line)
                pair = (str(review["user_id"]), str(review["item_id"]))
                if pair in allowed:
                    output.write(line)
                    found.add(pair)
                    written += 1
        if found != allowed:
            raise ValueError(f"Review coverage mismatch: {len(allowed - found)} fit pairs have no review")
        os.replace(temporary_path, output_path)
    finally:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
    return {"written_reviews": written, "distinct_fit_pairs": len(allowed)}


def _print_json(value: dict) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Shared Amazon Top-20 evaluation tools")
    commands = parser.add_subparsers(dest="command", required=True)

    split = commands.add_parser("split", help="Create deterministic validation CSVs")
    source = split.add_mutually_exclusive_group(required=True)
    source.add_argument("--train", help="Path to extracted train.csv")
    source.add_argument("--competition-zip", help="Path to original competition ZIP")
    split.add_argument("--out", required=True, help="Output directory")
    split.add_argument("--seed", type=int, default=SPLIT_SEED)
    split.add_argument("--fraction", type=float, default=HOLDOUT_FRACTION)
    split.add_argument("--overwrite", action="store_true")

    verify = commands.add_parser("verify-split", help="Check shared split file hashes")
    verify.add_argument("--dir", required=True)

    score = commands.add_parser("score", help="Validate and score validation predictions")
    score.add_argument("--pred", required=True)
    score.add_argument("--fit", required=True)
    score.add_argument("--truth", required=True)

    check = commands.add_parser("check", help="Validate a final test submission")
    check.add_argument("--pred", required=True)
    check.add_argument("--train", required=True)
    check.add_argument("--test", required=True)

    reviews = commands.add_parser("filter-reviews", help="Remove held-out reviews")
    reviews.add_argument("--fit", required=True)
    review_source = reviews.add_mutually_exclusive_group(required=True)
    review_source.add_argument("--reviews", help="Path to extracted train_reviews.jsonl")
    review_source.add_argument("--competition-zip", help="Path to original competition ZIP")
    reviews.add_argument("--out", required=True)

    args = parser.parse_args(argv)
    if args.command == "split":
        _print_json(make_split(args.train, args.competition_zip, args.out, args.seed, args.fraction, args.overwrite))
    elif args.command == "verify-split":
        manifest = verify_split(args.dir)
        _print_json({"split_id": manifest["split_id"], "verified": True})
    elif args.command == "score":
        seen, popularity = read_interactions(args.fit)
        truth = read_truth(args.truth)
        predictions = read_predictions(args.pred)
        catalog = set(popularity) | {item for items in truth.values() for item in items}
        validate_predictions(predictions, truth, seen, catalog)
        result = recall_at_20(predictions, truth)
        result["valid_submission_format"] = True
        _print_json(result)
    elif args.command == "check":
        seen, popularity = read_interactions(args.train)
        users = read_users(args.test)
        predictions = read_predictions(args.pred)
        validate_predictions(predictions, users, seen, set(popularity))
        _print_json({"valid": True, "users": len(users), "items_per_user": TOP_K})
    elif args.command == "filter-reviews":
        _print_json(filter_reviews(args.fit, args.out, args.reviews, args.competition_zip))
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (ValueError, FileExistsError, FileNotFoundError, KeyError, zipfile.BadZipFile) as error:
        print(f"ERROR: {error}", file=sys.stderr)
        sys.exit(1)


In [ ]:
%%writefile /kaggle/working/team_catboost_fulltrain_submit.py
#!/usr/bin/env python3
"""Reproduce the previously selected CatBoost depth-4 *pipeline* on official full train.

The earlier validation result used Group A (LightGCN/SimpleX/UltraGCN), Group B
(SVD PROXY), Group C (BM25-like PROXY), Group T (Masked-Item Transformer).
No D model was used. This program preserves that schema and does not claim to
reproduce teammate B/C/D real models.

Crucially: train the CatBoost model using an inner holdout of official train;
never fit it to unseen Kaggle test truth. Then refit ALL candidate generators on
full train and re-rank candidates for official test users. Only then export a
submission.csv with unseen unique Top-20 per user.
"""
from __future__ import annotations
import argparse
import csv
import hashlib
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch

import team_allsteps_model_pipeline as core
from team_allsteps_REAL_ablation import train_transformer

SEED=42
CAT_PARAMS={'iterations':250,'depth':4,'learning_rate':.05,'l2_leaf_reg':3,
            'loss_function':'PairLogit','random_seed':SEED,'verbose':False,'thread_count':4}
# Selected from the *original common validation*; fixed BEFORE new full-train run.
TRANSFORMER={'name':'T1_d128_L2_lr001','d':128,'layers':2,'heads':4,
             'epochs':50,'lr':.001,'batch':256,'maxlen':48,'eval_every':10}

def find_official_data(root:Path):
    candidates=[]
    for train in root.rglob('train.csv'):
        test=train.with_name('test.csv')
        if not test.is_file():continue
        try:
            with train.open(encoding='utf8',newline='') as f:
                fields=set(csv.DictReader(f).fieldnames or [])
            with test.open(encoding='utf8',newline='') as f:
                testfields=set(csv.DictReader(f).fieldnames or [])
            if {'user_id','item_id'}.issubset(fields) and 'user_id' in testfields:
                candidates.append((train,test))
        except Exception: pass
    if len(candidates)!=1:
        raise RuntimeError(f'공식 train.csv/test.csv 쌍을 정확히 하나 찾지 못했습니다. 연결된 데이터셋과 중복 파일을 확인하세요. candidates={candidates}')
    return candidates[0]

def main(args):
    import teamkit
    from catboost import CatBoostRanker

    work=Path(args.work); work.mkdir(parents=True,exist_ok=True)
    source_dir=Path(args.scripts)
    train,test = find_official_data(Path(args.input_root)) if not args.train else (Path(args.train),Path(args.test))
    assert train.is_file() and test.is_file(),(train,test)
    users=teamkit.read_users(test)
    seen,events,pop=core.load_interactions(train)
    items=set(pop)
    if not users or len(items)<20:raise RuntimeError('Invalid official dataset or item count')
    print('== OFFICIAL DATA ==',flush=True)
    print('Train:',train,'Test:',test,'users:',len(users),'items:',len(items),flush=True)
    print('Groups: A=real own implementations; B=SVD PROXY; C=BM25-like PROXY; T=masked-set Transformer; D=ABSENT',flush=True)
    print('Validation-selected CatBoost:',CAT_PARAMS,flush=True)
    print('Validation-selected Transformer epochs:',TRANSFORMER['epochs'],flush=True)

    # Split the ORIGINAL FULL TRAIN to generate out-of-sample ranking labels.
    inner=work/'ranker_inner';inner.mkdir(parents=True,exist_ok=True)
    inner_fit=inner/'train.csv';inner_truth_file=inner/'truth.csv';inner_users_file=inner/'users.csv'
    split_stamp=inner/'source_stamp.json'
    fingerprint={'train_sha256':core.sha256(train),'seed':47,'fraction':0.18,'version':'catboost_fulltrain_v1'}
    if not inner_fit.exists() or not inner_truth_file.exists() or not inner_users_file.exists() or not split_stamp.exists() or json.loads(split_stamp.read_text())!=fingerprint:
        core.inner_split(train,inner_fit,inner_truth_file,inner_users_file,seed=47,holdout=.18)
        core.atomic_json(split_stamp,fingerprint)
    inner_users=teamkit.read_users(inner_users_file)
    inner_truth=teamkit.read_truth(inner_truth_file)
    print('Inner train users:',len(inner_users),'labels:',sum(map(len,inner_truth.values())),flush=True)

    native_profile='SMOKE' if args.smoke else 'FULL'
    device=args.device
    print('\n== STEP A: fit candidate generators on INNER training split ==',flush=True)
    mi,seen_i,pop_i=core.prepare_native('catboost_inner',inner_fit,inner_users,work,native_profile,device,
                '',source_dir,False,False,'','')
    tcfg=dict(TRANSFORMER)
    if args.smoke:tcfg.update(name='T_smoke',epochs=2,d=32,layers=1,batch=8,maxlen=16,eval_every=1)
    ti,_=train_transformer(inner_fit,inner_users,inner_truth,work/'transformer_inner',tcfg,device,select_best=False)
    mi['T']=ti
    assert sorted(mi)==['A','B','C','T'],sorted(mi)

    print('\n== STEP B: train CAT_depth4 on INNER holdout ==',flush=True)
    fallback_i=[item for item,_ in pop_i.most_common()]
    Xi,yi,ui,iti,gi,names_i=core.make_ranking_features(mi,inner_users,seen_i,pop_i,fallback_i,
            max_users=args.ranker_train_users,seed=SEED,truth=inner_truth,positive_only=True)
    print('Ranker training rows:',len(yi),'positive labels:',int(yi.sum()),'groups:',len(gi),'features:',Xi.shape[1],flush=True)
    if len(gi)<10 or int(yi.sum())<10:raise RuntimeError('Not enough inner positive training candidates for CatBoost')
    cat=CatBoostRanker(**({**CAT_PARAMS,**({'iterations':10,'depth':3} if args.smoke else {})}))
    cat.fit(Xi,yi,group_id=ui,verbose=False)
    del Xi,yi,ui,iti,gi
    cat_model=work/'catboost_depth4_fulltrain.cbm';cat.save_model(str(cat_model))
    print('Saved trained ranker:',cat_model,flush=True)
    del mi,ti
    if device.startswith('cuda'):torch.cuda.empty_cache()

    print('\n== STEP C: refit candidate generators on ENTIRE official train.csv ==',flush=True)
    mf,seen_f,pop_f=core.prepare_native('catboost_full',train,users,work,native_profile,device,
            '',source_dir,False,False,'','')
    tf,_=train_transformer(train,users,{},work/'transformer_full',tcfg,device,select_best=False)
    mf['T']=tf
    if sorted(mf)!=names_i:raise RuntimeError(f'Feature schema mismatch, inner={names_i} outer={sorted(mf)}')
    for name,pred in mf.items():core.validate_pred(pred,users,seen_f,items)

    print('\n== STEP D: rerank official Test-user candidates and export Top-20 ==',flush=True)
    fallback_f=[item for item,_ in pop_f.most_common()]
    Xf,_,uf,itf,gf,names_f=core.make_ranking_features(mf,users,seen_f,pop_f,fallback_f,truth=None)
    if names_f!=names_i:raise RuntimeError('Wrong feature order')
    print('Test candidate rows:',len(Xf),'target users:',len(users),'feature groups:',names_f,flush=True)
    pr=core.predict_rerank(cat,Xf,uf,itf,users,seen_f,fallback_f)
    core.validate_pred(pr,users,seen_f,items,k=20)
    sub=Path(args.submission)
    sub.parent.mkdir(parents=True,exist_ok=True)
    core.write_pred(sub,pr,users)
    # Check the exact official submission format with original teamkit utility.
    tp=teamkit.read_predictions(sub)
    teamkit.validate_predictions(tp,users,seen_f,items)
    print('✅ Submit CSV:',sub,'users:',len(users),'size_bytes:',sub.stat().st_size,flush=True)

    meta={'input_train':str(train),'input_test':str(test),'train_sha256':core.sha256(train),
      'team_member_note':'Group B=SVD proxy, Group C=BM25-like proxy, Group D ABSENT; NOT full 4-person ensemble.',
      'base_a':{'lightgcn_epochs':(2 if args.smoke else 575),'simplex_epochs':(2 if args.smoke else 650),'ultragcn_epochs':(2 if args.smoke else 80)},
      'transformer_config':tcfg,'catboost_params':cat.get_params(),
      'ranker_training':'Inner split of FULL original train, leakage-safe for Kaggle test','feature_models':names_f,
      'full_test_users':len(users),'submission':str(sub),'validation_reference':{'experiment':'CAT_depth4','recall_at_20':0.04624893280113251},
      'IMPORTANT':'Validation score is not the Kaggle test score. This is NOT the teammate D model.'}
    core.atomic_json(work/'submission_metadata.json',meta)
    bundle=Path(args.bundle)
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
        z.write(sub,arcname='submission_catboost.csv');z.write(work/'submission_metadata.json',arcname='submission_metadata.json')
    print('✅ Download ZIP:',bundle,flush=True)

def parser():
    p=argparse.ArgumentParser()
    p.add_argument('--input-root',default='/kaggle/input');p.add_argument('--train',default='');p.add_argument('--test',default='')
    p.add_argument('--work',default='/kaggle/working/catboost_submit')
    p.add_argument('--scripts',default='/kaggle/working')
    p.add_argument('--submission',default='/kaggle/working/submission_catboost.csv')
    p.add_argument('--bundle',default='/kaggle/working/catboost_submission.zip')
    p.add_argument('--device',default='cuda:0' if torch.cuda.is_available() else 'cpu')
    p.add_argument('--ranker-train-users',type=int,default=18000)
    p.add_argument('--smoke',action='store_true',help='tiny CPU tests only; NOT for submission')
    return p.parse_args()
if __name__=='__main__':main(parser())


## 2. 대회 원본 Train/Test 파일 자동 확인

오직 원본 `train.csv`와 `test.csv`가 필요하다. 팀 공통 Validation ZIP은 이번 노트북에서 사용하지 않는다.

In [ ]:
import sys
sys.path.insert(0,str(WORK))
from team_catboost_fulltrain_submit import find_official_data
TRAIN,TEST=find_official_data(DATA)
print('✅ 원본 train:',TRAIN)
print('✅ 원본 test:',TEST)
print('Train size (MB):',round(TRAIN.stat().st_size/1024**2,2))
print('Test size (MB):',round(TEST.stat().st_size/1024**2,2))


## 3. 전체 학습 및 CatBoost 제출 예측 생성

다음 작업이 모두 실행된다: (1) 원본 Train에서 내부 Holdout 생성 (2) A/B/C/T 모델을 내부 Train으로 학습해 CatBoost 재랭킹 학습 데이터 구성 (3) CatBoost Depth-4 학습 (4) **원본 전체 Train으로 A/B/C/T 재학습** (5) Test Top-20 생성·검증.

**주의:** `--smoke` 옵션은 제출용으로 사용하지 않는다.

In [ ]:
import subprocess,sys,os
cmd=[sys.executable,str(WORK/'team_catboost_fulltrain_submit.py'),
    '--train',str(TRAIN),'--test',str(TEST),
    '--scripts',str(WORK),
    '--work',str(WORK/'catboost_submit'),
    '--submission',str(WORK/'submission_catboost.csv'),
    '--bundle',str(WORK/'catboost_submission.zip'),
    '--device','cuda:0',
    '--ranker-train-users','18000']
print('Starting full-train CatBoost submission pipeline:', ' '.join(cmd),flush=True)
with (WORK/'catboost_fulltrain.log').open('w',encoding='utf8') as log:
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env={**os.environ,'PYTHONUNBUFFERED':'1'})
    for line in process.stdout:
        print(line,end='')
        log.write(line);log.flush()
    result=process.wait()
if result!=0:
    raise RuntimeError(f'최종 학습 실행 실패 (exit {result}). catboost_fulltrain.log의 오류 확인')
print('✅ 전체 학습 및 제출 CSV 생성 완료')


## 4. 제출 CSV 검사 및 다운로드

In [ ]:
import json, zipfile, pandas as pd
from IPython.display import display,FileLink
sub=WORK/'submission_catboost.csv'
bundle=WORK/'catboost_submission.zip'
meta=WORK/'catboost_submit'/'submission_metadata.json'
assert sub.exists() and sub.stat().st_size>0,'제출용 CSV 파일이 생성되지 않았습니다.'
assert bundle.exists() and bundle.stat().st_size>0,'다운로드 ZIP이 생성되지 않았습니다.'
report=json.loads(meta.read_text(encoding='utf8'))
df=pd.read_csv(sub)
assert list(df.columns)==['user_id','item_id'],df.columns.tolist()
assert len(df)==report['full_test_users']
assert df['user_id'].nunique()==len(df)
assert df['item_id'].apply(lambda s: len(str(s).split())==20 and len(set(str(s).split()))==20).all()
print('✅ SUBMISSION READY')
print('제출 CSV:',sub,'(',len(df),'users )')
print('ZIP:',bundle)
print('학습 메타:',meta)
display(df.head(3))
print('Output 목록 또는 아래 파일 링크에서 다운로드:')
display(FileLink(str(sub)))
display(FileLink(str(bundle)))
print('Kaggle 파일 다운로드에서 404가 뜨면 오른쪽 Output의 파일 또는 저장된 Notebook Version의 Output에서 ZIP을 다운로드하세요.')


### Kaggle에 제출하는 파일

`submission_catboost.csv` **하나만** `Submit Predictions`에 업로드한다. `.zip`은 CSV를 내려받기 편하도록 만든 묶음 파일이므로 그대로 제출하지 않는다. ZIP 안의 `submission_catboost.csv`를 압축 해제하여 제출할 수 있다.

**제출 Description 예시:** `CatBoost Rerank d4 i250 lr0.05 | A+proxyB+proxyC+T | full train`

제출 후 실제 Public Test 점수를 보고 이전 앙상블 0.4724(점수 표기 재확인 필요)와 비교한다.